In [ ]:
# ============================================================
# Background Studies — gamma vs muoni LNGS
# Logica di rivelazione: A∪B (almeno un PMT con segnale)
# ============================================================

import uproot
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os
import warnings
from scipy.signal import butter

warnings.filterwarnings('ignore')

# ============================================================
# === CONFIGURAZIONE =========================================
# ============================================================
DATA_DIR   = "/Users/benussi/Testbeam2026_WC_unified/data"
OUTPUT_DIR = "/Users/benussi/Testbeam2026_WC_unified/Analysis_script_background/output"

# === GEOMETRIE PER GLI STUDI FINALI ===
# Studi finali sulle due geometrie candidate, ENTRAMBE ALLA PARI: WBC2 e WBC3.
# La geometria single originale resta COMMENTATA (storica) per confronto e per
# giustificare la scelta finale: per riattivarla, aggiungere "single" a GEO_LIST
# e decommentare la riga corrispondente in GEO_STYLE.
GEO_LIST  = ["single_wbc2", "single_wbc3"]
GEO_MAIN  = "single_wbc3"     # geometria che popola `results` per le celle legacy (results[src])
GEO_STYLE = {
    # colori ben distinti: WBC2 = toni caldi (rosso/arancio), WBC3 = toni freddi (blu/viola)
    "single_wbc2": {"tag": "WBC2", "color_mu": "#d62728", "color_g": "#ff7f0e", "ls": "-",  "marker": "o"},
    "single_wbc3": {"tag": "WBC3", "color_mu": "#1f77b4", "color_g": "#9467bd", "ls": "--", "marker": "s"},
    # "single":    {"tag": "single (orig.)", "color_mu": "#7f7f7f", "color_g": "#c7c7c7", "ls": ":", "marker": "^"},  # storico
}

def _make_sources(geo):
    st = GEO_STYLE[geo]
    return {
        "gamma":   {"file": f"sim_{geo}_gamma_X0mm_Y0mm_Th0deg.root",
                    "color": st["color_g"],  "label": rf"$\gamma$ {st['tag']}",      "ls": st["ls"]},
        "mu_lngs": {"file": f"sim_{geo}_mu_lngs_X0mm_Y0mm_Th0deg.root",
                    "color": st["color_mu"], "label": rf"$\mu^-$ LNGS {st['tag']}",  "ls": st["ls"]},
    }

SOURCES = _make_sources(GEO_MAIN)

CH_A, CH_B = 0, 1

# --- Digitizer ---
SAMPLE_RATE_GS    = 2.5
DT_NS             = 1.0 / SAMPLE_RATE_GS
RECORD_LENGTH     = 1024
ADC_BITS          = 12
V_PP_MV           = 1000.0
NOISE_RMS_ADC     = 2.0   # RMS per campione in ADC (~0.5 mV);
                          # ⚠ ancorare al valore misurato della DAQ reale
                          # in carica: σ_Q = 2·√150 ≈ 24.5 ADC·samples ≈ 1.2 SPE (H14603!)
TRIGGER_OFFSET_NS = 150.0

# --- PMT: Hamamatsu H10721-210 (ultra-bialkali, HV integrato) ---
# (H14603-103 fuori produzione; SiPM tenuto nella cella readout per confronto)
G            = 2.0e6
R            = 50.0
e_charge     = 1.602e-19
TTS_FWHM_NS  = 0.3
TTS_SIGMA_NS = TTS_FWHM_NS / 2.355
SPE_RES      = 0.35

# --- Cavo: 10m RG-58 ---
CABLE_LENGTH_M = 10.0
ATTENUATION_DB = (15.0 / 100.0) * CABLE_LENGTH_M
TRANSMISSION   = 10.0 ** (-ATTENUATION_DB / 20.0)

tr_pmt      = 0.6
tr_cable    = 0.14 * CABLE_LENGTH_M
tr_total    = np.sqrt(tr_pmt**2 + tr_cable**2)
sigma_total = tr_total / 2.22

amp_SPE_mV = (-(e_charge * G * R) /
              (sigma_total * 1e-9 * np.sqrt(2 * np.pi)) * 1e3 * TRANSMISSION)
LSB_MV = V_PP_MV / (2**ADC_BITS)

# Carica di 1 SPE in ADC·samples
q_SPE = abs(amp_SPE_mV) * sigma_total / DT_NS * np.sqrt(2 * np.pi) / LSB_MV

# --- Gate di integrazione ---
GATE_START_NS = 140.0
GATE_END_NS   = 200.0

# --- Soglia nominale (per tabelle e fig8) ---
THR_PEAK_MV = -1.0 * abs(amp_SPE_mV)   # ~1 SPE in mV (nuovo PMT, ~-8 mV)
THR_CHARGE  =  1.0 * q_SPE             # ~1 SPE in ADC·samples (nuovo PMT, ~138)

# --- Plot config ---
HIST_BINS       = 60
MAX_CHARGE      = None
MAX_DEBUG_PLOTS = 2   # eventi per sorgente nel confronto waveform 3 sensori

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"amp_SPE (attenuato): {amp_SPE_mV:.2f} mV")
print(f"q_SPE:               {q_SPE:.1f} ADC·samples")
print(f"Gate carica:         [{GATE_START_NS:.0f}, {GATE_END_NS:.0f}] ns")
print(f"Logica rivelazione:  A∪B  (modulo acceso se almeno un PMT ha segnale)")
print(f"  Picco:   peak_A < {THR_PEAK_MV:.0f} mV  OR  peak_B < {THR_PEAK_MV:.0f} mV")
print(f"  Carica:  max(QA,QB) > {THR_CHARGE:.0f} ADC·samples")

# ============================================================
# === HELPERS ================================================
# ============================================================
time_axis = np.arange(RECORD_LENGTH) * DT_NS
_gate_i0  = int(GATE_START_NS / DT_NS)
_gate_i1  = min(RECORD_LENGTH - 1, int(GATE_END_NS / DT_NS))

def get_qe(wl_nm):
    wl = np.array([200,250,300,350,380,400,420,450,500,550,600,650,700])
    qe = np.array([0.20,0.30,0.38,0.42,0.43,0.43,0.42,0.38,0.27,0.14,0.05,0.010,0.00])  # UBA H10721-210
    return np.interp(wl_nm, wl, qe, left=0., right=0.)

def digitize_pmt(t_accepted):
    noise = np.random.normal(0., NOISE_RMS_ADC * (V_PP_MV / 2**ADC_BITS), RECORD_LENGTH)
    if len(t_accepted) == 0:
        return noise
    jitter = np.random.normal(0., TTS_SIGMA_NS, len(t_accepted))
    gains  = np.maximum(0., np.random.normal(1., SPE_RES, len(t_accepted)))
    t_s    = t_accepted + TRIGGER_OFFSET_NS + jitter
    dt_mat = time_axis[:, None] - t_s[None, :]
    voltage = amp_SPE_mV * (np.exp(-0.5 * (dt_mat / sigma_total)**2) @ gains)
    return voltage + np.random.normal(0., NOISE_RMS_ADC * (V_PP_MV / 2**ADC_BITS), RECORD_LENGTH)

def charge(wf_mv, i0=_gate_i0, i1=_gate_i1):
    return float(-np.trapezoid(wf_mv[i0:i1]) / LSB_MV)

# ============================================================
# === DEBUG waveform — confronto 3 sensori sugli stessi eventi
#     (H14603-103 vecchio | H10721-210 | SiPM S14160-6075HS)
#     Stessi fotoni MC, stessi numeri casuali di accettazione
#     (u condivisi), rumore NOISE_RMS_ADC incluso.
# ============================================================
if MAX_DEBUG_PLOTS > 0:
    DBG_SENSORS = {
        "H14603-103 (old)": dict(type="pmt", gain=3.0e5, spe_res=0.40, tts=TTS_SIGMA_NS,
            wl =[200,250,300,350,380,400,420,450,500,550,600,650,700],
            eff=[0.18,0.25,0.30,0.34,0.35,0.34,0.32,0.28,0.18,0.08,0.03,0.005,0.0]),
        "H10721-210": dict(type="pmt", gain=2.0e6, spe_res=0.35, tts=TTS_SIGMA_NS,
            wl =[200,250,300,350,380,400,420,450,500,550,600,650,700],
            eff=[0.20,0.30,0.38,0.42,0.43,0.43,0.42,0.38,0.27,0.14,0.05,0.01,0.0]),
        "S14160-6075HS": dict(type="sipm", gain=5.0e6, spe_res=0.10, tts=0.15,
            wl =[280,300,350,400,450,500,550,600,650,700,800,900],
            eff=[0.10,0.18,0.32,0.44,0.50,0.46,0.40,0.33,0.27,0.21,0.10,0.04],
            dcr=2.0e6, p_ct=0.07, tau_r=1.0, tau_f=25.0),
    }
    _rng_dbg = np.random.default_rng(2026)

    def _dbg_wf(t_raw, wl_raw, u_raw, scfg):
        """waveform (mV) e n. fotoni rivelati per un canale/sensore"""
        noise = _rng_dbg.normal(0., NOISE_RMS_ADC * LSB_MV, RECORD_LENGTH)
        if len(t_raw):
            acc = u_raw < np.interp(wl_raw, scfg["wl"], scfg["eff"], left=0., right=0.)
            t_acc = t_raw[acc]
        else:
            t_acc = np.array([])
        g = np.maximum(0., _rng_dbg.normal(1., scfg["spe_res"], len(t_acc)))
        if scfg["type"] == "pmt":
            if len(t_acc) == 0:
                return noise, (0, 0)
            ts = t_acc + TRIGGER_OFFSET_NS + _rng_dbg.normal(0., scfg["tts"], len(t_acc))
            dt = time_axis[:, None] - ts[None, :]
            amp = amp_SPE_mV * scfg["gain"] / 3.0e5
            return amp * (np.exp(-0.5*(dt/sigma_total)**2) @ g) + noise, (len(t_acc), 0)
        # SiPM: doppio esponenziale + dark counts + crosstalk
        n_dark = _rng_dbg.poisson(scfg["dcr"] * RECORD_LENGTH * DT_NS * 1e-9)
        ts = np.concatenate([t_acc + TRIGGER_OFFSET_NS +
                             _rng_dbg.normal(0., scfg["tts"], len(t_acc)),
                             _rng_dbg.uniform(0, RECORD_LENGTH*DT_NS, n_dark)])
        gg = np.concatenate([g, np.maximum(0., _rng_dbg.normal(1., scfg["spe_res"], n_dark))])
        if len(ts) == 0:
            return noise, (0, n_dark)
        gg = gg * (1.0 + (_rng_dbg.random(len(gg)) < scfg["p_ct"]).astype(float))
        C = scfg["gain"]*e_charge*R*TRANSMISSION*1e12 / (scfg["tau_f"]-scfg["tau_r"])
        dt = time_axis[:, None] - ts[None, :]
        sh = np.where(dt >= 0, np.exp(-np.clip(dt,0,None)/scfg["tau_f"])
                             - np.exp(-np.clip(dt,0,None)/scfg["tau_r"]), 0.0)
        return -C * (sh @ gg) + noise, (len(t_acc), n_dark)

    for src_name, cfg in SOURCES.items():
        fpath = os.path.join(DATA_DIR, cfg["file"])
        if not os.path.exists(fpath):
            continue
        with uproot.open(fpath) as f:
            df_dbg = f["Fotoni"].arrays(
                ["EventID","Arrival_Time_ns","PMT_ID","E_Hit_eV"], library="pd")
        for ev_id in np.sort(df_dbg["EventID"].unique())[:MAX_DEBUG_PLOTS]:
            hits = df_dbg[df_dbg["EventID"] == ev_id]
            fig, axes = plt.subplots(3, 2, figsize=(14, 9), sharex=True)
            fig.suptitle(f"[{GEO_STYLE[GEO_MAIN]['tag']}]  {src_name} — EventID {ev_id} — waveform comparison "
                         f"(noise {NOISE_RMS_ADC:.0f} ADC)", fontsize=12)
            for ch in (CH_A, CH_B):
                ph_ch = hits[hits["PMT_ID"] == ch]
                if ph_ch.empty:
                    t_raw = wl_raw = u_raw = np.array([])
                else:
                    t_raw = ph_ch["Arrival_Time_ns"].values
                    wl_raw = 1240.0 / ph_ch["E_Hit_eV"].values
                    u_raw = _rng_dbg.random(len(t_raw))   # condivisi tra sensori
                for row, (sname, scfg) in enumerate(DBG_SENSORS.items()):
                    ax = axes[row][ch]
                    wf_dbg, (npe, ndk) = _dbg_wf(t_raw, wl_raw, u_raw, scfg)
                    ax.plot(time_axis, wf_dbg, color="darkcyan", lw=0.9)
                    ax.set_ylabel(f"{sname}\n(mV)", fontsize=9)
                    _lbl = f"{npe} p.e. det.  ({len(t_raw)} ph MC)"
                    if scfg["type"] == "sipm": _lbl += f"  +{ndk} dark"
                    ax.text(0.98, 0.06, _lbl,
                            transform=ax.transAxes, ha="right", fontsize=8)
                    if row == 0: ax.set_title(f"ch {ch}", fontsize=10)
                    if row == 2: ax.set_xlabel("t (ns)", fontsize=10)
                    ax.set_xlim(80, 350)
                    ax.grid(True, alpha=0.25)
            plt.tight_layout(); plt.show()

# ============================================================
# === LOOP PRINCIPALE ========================================
# ============================================================
def digitize_sources(sources):
    """Digitizza le sorgenti di UNA geometria -> dict results[src_name]."""
    results = {}
    for src_name, cfg in sources.items():
        fpath = os.path.join(DATA_DIR, cfg["file"])
        if not os.path.exists(fpath):
            print(f"[SKIP] {cfg['file']} non trovato")
            continue

        print(f"\n--- {src_name} | {cfg['file']} ---")
        with uproot.open(fpath) as f:
            t_ev  = f["Eventi"]
            n_gen = int(t_ev.num_entries)
            df    = f["Fotoni"].arrays(
                ["EventID","Arrival_Time_ns","PMT_ID","E_Hit_eV"], library="pd")

        if df.empty:
            print("  Nessun fotone — skip.")
            continue

        ev_ids = np.sort(df["EventID"].unique())
        print(f"  N eventi generati:       {n_gen}")
        print(f"  N eventi con fotoni PMT: {len(ev_ids)}")

        qa_list, qb_list         = [], []
        peak_a_list, peak_b_list = [], []

        for ev_id in ev_ids:
            hits = df[df["EventID"] == ev_id]
            wf = {}
            for ch in (CH_A, CH_B):
                ph_ch = hits[hits["PMT_ID"] == ch]
                if ph_ch.empty:
                    t_acc = np.array([])
                else:
                    wl_nm = 1240.0 / ph_ch["E_Hit_eV"].values
                    t_acc = ph_ch["Arrival_Time_ns"].values[
                        np.random.rand(len(wl_nm)) < get_qe(wl_nm)]
                wf[ch] = digitize_pmt(t_acc)
            qa_list.append(charge(wf[CH_A]))
            qb_list.append(charge(wf[CH_B]))
            peak_a_list.append(float(wf[CH_A][_gate_i0:_gate_i1].min()))
            peak_b_list.append(float(wf[CH_B][_gate_i0:_gate_i1].min()))

        qa = np.array(qa_list); qb = np.array(qb_list)
        peak_a = np.array(peak_a_list); peak_b = np.array(peak_b_list)

        # Fotoni Geant4 al PMT (pre-QE) per canale e per evento
        _df_ab = df[df["PMT_ID"].isin((CH_A, CH_B))]
        _ph_A  = _df_ab[_df_ab["PMT_ID"] == CH_A].groupby("EventID")["PMT_ID"].count()
        _ph_B  = _df_ab[_df_ab["PMT_ID"] == CH_B].groupby("EventID")["PMT_ID"].count()
        _ev_union   = _ph_A.index.union(_ph_B.index)
        n_ph_A_ev   = _ph_A.reindex(_ev_union, fill_value=0).values
        n_ph_B_ev   = _ph_B.reindex(_ev_union, fill_value=0).values
        ev_ph_count = n_ph_A_ev + n_ph_B_ev

        results[src_name] = {
            "qa": qa, "qb": qb,
            "peak_a": peak_a, "peak_b": peak_b,
            "n_ph_pmt": ev_ph_count,
            "n_ph_A":   n_ph_A_ev,
            "n_ph_B":   n_ph_B_ev,
            "n_gen":    n_gen,
            "n_hits":   len(ev_ids),
            "cfg":      cfg,
        }
        print(f"  max(QA,QB) medio = {np.maximum(qa,qb).mean():.1f}")
        print(f"  min(pA,pB) medio = {np.minimum(peak_a,peak_b).mean():.2f} mV")
    return results

# Digitizza ENTRAMBE le geometrie finali (WBC2 e WBC3) -> RESULTS_GEO[geo][src]
RESULTS_GEO = {}
for _geo in GEO_LIST:
    print(f"\n========== GEOMETRIA {GEO_STYLE[_geo]['tag']} ({_geo}) ==========")
    RESULTS_GEO[_geo] = digitize_sources(_make_sources(_geo))

results = RESULTS_GEO[GEO_MAIN]     # back-compat: celle legacy usano `results[src]` (= WBC3)

if not results:
    print("\nNessun file trovato.")
    raise SystemExit(0)

# ============================================================
# === PLOT: distribuzioni di carica ==========================
# ============================================================
all_qa = np.concatenate([r["qa"] for r in results.values()])
all_qb = np.concatenate([r["qb"] for r in results.values()])
q_min  = min(all_qa.min(), all_qb.min())
q_max  = MAX_CHARGE if MAX_CHARGE else max(np.percentile(all_qa, 99.5),
                                           np.percentile(all_qb, 99.5))
bins   = np.linspace(q_min, q_max, HIST_BINS + 1)

for fig_log, suffix in [(False, ""), (True, "_log")]:
    fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"PMT charge distribution — gamma vs LNGS muons  (A∪B)"
                 + (" [log]" if fig_log else ""), fontsize=13)
    for src_name, res in results.items():
        col = res["cfg"]["color"]
        lbl = f"{res['cfg']['label']}  (n={res['n_hits']} / {res['n_gen']} gen)"
        ax_a.hist(res["qa"], bins=bins, density=True, color=col, alpha=0.55,
                  histtype="stepfilled", edgecolor=col, linewidth=1.2, label=lbl)
        ax_b.hist(res["qb"], bins=bins, density=True, color=col, alpha=0.55,
                  histtype="stepfilled", edgecolor=col, linewidth=1.2, label=lbl)
    for ax, ch_lbl in [(ax_a, f"PMT A — Left (ch {CH_A})"),
                       (ax_b, f"PMT B — Right (ch {CH_B})")]:
        if fig_log: ax.set_yscale("log")
        ax.set_xlabel("Charge (ADC·samples, 1 SPE ≈ 21)", fontsize=11)
        ax.set_ylabel("Probability density", fontsize=11)
        ax.set_title(ch_lbl, fontsize=11)
        ax.legend(fontsize=9); ax.grid(True, alpha=0.3, which="both" if fig_log else "major")
        ax.axvline(0, color="gray", lw=0.8, ls="--")
    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, f"charge_dist_gamma_vs_mu_lngs{suffix}.pdf"), bbox_inches="tight")
    plt.show()

# ============================================================
# === RIEPILOGO ===============================================
# ============================================================
print("\n" + "="*65)
print(f"{'Sorgente':12} | {'N gen':>10} | {'N fotoni PMT':>13} | {'QA media':>10}")
print("-"*65)
for src_name, res in results.items():
    print(f"{src_name:12} | {res['n_gen']:>10d} | {res['n_hits']:>13d} | {res['qa'].mean():>10.1f}")
print("="*65)

# ============================================================
# === PLOT: fotoni Cherenkov ai PMT ==========================
# ============================================================
fig3, (ax3_lin, ax3_log) = plt.subplots(1, 2, figsize=(14, 5))
fig3.suptitle("Cherenkov photons at PMTs — gamma vs LNGS muons  (A∪B)", fontsize=12)

_ph_max = max(int(res["n_ph_pmt"].max()) for res in results.values() if len(res["n_ph_pmt"]) > 0)
_ph_bins_lin  = np.linspace(0.5, _ph_max + 0.5, min(_ph_max + 1, 81))
_ph_bins_logx = np.logspace(np.log10(0.8), np.log10(_ph_max + 1), 60)

for res in results.values():
    arr = res["n_ph_pmt"]; col = res["cfg"]["color"]
    lbl = f"{res['cfg']['label']}  (n={len(arr)})"
    ax3_lin.hist(arr, bins=_ph_bins_logx, density=True, color=col, alpha=0.55,
                 histtype="stepfilled", edgecolor=col, linewidth=1.0, label=lbl)
    ax3_log.hist(arr, bins=_ph_bins_lin, density=True, color=col, alpha=0.55,
                 histtype="stepfilled", edgecolor=col, linewidth=1.0, label=lbl)

ax3_lin.set_xscale("log"); ax3_lin.set_yscale("log")
ax3_lin.set_xlabel("N photons at PMT (ch A + ch B)", fontsize=11)
ax3_lin.set_ylabel("Probability density", fontsize=11)
ax3_lin.set_title("Log-log scale", fontsize=11)
ax3_lin.legend(fontsize=9); ax3_lin.grid(True, alpha=0.3, which="both")

ax3_log.set_yscale("log")
ax3_log.set_xlabel("N photons at PMT (ch A + ch B)", fontsize=11)
ax3_log.set_ylabel("Probability density", fontsize=11)
ax3_log.set_title("Log-Y scale", fontsize=11)
ax3_log.legend(fontsize=9); ax3_log.grid(True, alpha=0.3, which="both")
plt.tight_layout()
fig3.savefig(os.path.join(OUTPUT_DIR, "n_photons_pmt_gamma_vs_mu_lngs.pdf"), bbox_inches="tight")
plt.show()

# ============================================================
# === PLOT: efficienza vs soglia picco (A∪B) =================
#   Condizione A∪B: min(peak_A, peak_B) < −|thr|
#   X-axis: |soglia| da 0.5 a 10 mV
#   Denominatore: n_hits = eventi con ≥1 fotone al PMT
# ============================================================
THR_AMP = np.linspace(0.5, 8.0*abs(amp_SPE_mV), 400)  # da 0.5 mV a ~8 SPE (nuovo PMT)

fig4, ax4 = plt.subplots(figsize=(9, 6))
fig4.suptitle(
    "Efficiency vs peak threshold  (A∪B)\n"
    r"eff = N[min(peak$_A$, peak$_B$) $<$ $-$|thr|] / $N_{hits}$   (events with ≥1 photon at PMT)",
    fontsize=12)

for src_name, res in results.items():
    pmin       = np.minimum(res["peak_a"], res["peak_b"])
    n_hits_src = res["n_hits"]
    effs = np.array([100.0 * (pmin < -thr).sum() / n_hits_src for thr in THR_AMP])
    ax4.plot(THR_AMP, effs, color=res["cfg"]["color"], lw=2.0,
             label=f"{res['cfg']['label']}  (N hits={n_hits_src})")

ax4.axvline(abs(THR_PEAK_MV), color="gray", lw=1.2, ls="--",
            label=f"Nominal threshold = {abs(THR_PEAK_MV):.0f} mV")

for k in [1, 2, 3]:
    amp_k = abs(k * amp_SPE_mV)
    if amp_k <= 10.0:
        ax4.axvline(amp_k, color="green", lw=1.2, ls=":",
                    label=f"{k} SPE = {amp_k:.2f} mV" if k == 1 else f"{k} SPE")

ax4.set_xlabel(r"|Peak threshold|  (mV)  —  signal if min(peak$_A$,peak$_B$) $<$ $-$|thr|",
               fontsize=11)
ax4.set_ylabel(r"Efficiency  (%)", fontsize=12)
ax4.set_xlim(0.5, 10.0); ax4.set_yscale("log"); ax4.set_ylim(0.01, 110)
ax4.axhline(100, color="lightgray", lw=0.8, ls="--")
ax4.legend(fontsize=10); ax4.grid(True, alpha=0.3, which="both")
plt.tight_layout()
fig4.savefig(os.path.join(OUTPUT_DIR, "efficiency_vs_threshold_peak_mV.pdf"), bbox_inches="tight")
plt.show()

# ============================================================
# === PLOT: efficienza vs soglia carica (A∪B) ================
#   Condizione A∪B: max(QA, QB) > ThrQ
#   X-axis: ThrQ da 0 a 10*q_SPE (in ADC·samples)
#   Asse superiore in unità di SPE
#   Denominatore: n_hits = eventi con ≥1 fotone al PMT
# ============================================================
THR_Q_MAX   = 10.0 * q_SPE                       # 10 SPE in ADC·samples
THR_Q_RANGE = np.linspace(0, THR_Q_MAX, 500)

fig_q, ax_q = plt.subplots(figsize=(9, 6))
fig_q.suptitle(
    "Efficiency vs charge threshold  (A∪B)\n"
    r"eff = N[max(QA,QB) $>$ ThrQ] / $N_{hits}$   (events with ≥1 photon at PMT)",
    fontsize=12)

for src_name, res in results.items():
    q_mx       = np.maximum(res["qa"], res["qb"])
    n_hits_src = res["n_hits"]
    effs = np.array([100.0 * (q_mx > thr).sum() / n_hits_src for thr in THR_Q_RANGE])
    ax_q.plot(THR_Q_RANGE, effs, color=res["cfg"]["color"], lw=2.0,
              label=f"{res['cfg']['label']}  (N hits={n_hits_src})")

# Marca soglia nominale corrente
ax_q.axvline(THR_CHARGE, color="gray", lw=1.2, ls="--",
             label=f"Nominal threshold = {THR_CHARGE:.0f} ADC·samples")

# Marca posizioni degli SPE
for k in [1, 2, 3, 5, 10]:
    q_k = k * q_SPE
    if q_k <= THR_Q_MAX:
        ax_q.axvline(q_k, color="green", lw=1.2, ls=":",
                     label=f"{k} SPE = {q_k:.0f} ADC·samples" if k == 1 else f"{k} SPE")

ax_q.set_xlabel(r"ThrQ  (ADC·samples)  —  signal if max(QA,QB) $>$ ThrQ", fontsize=11)
ax_q.set_ylabel(r"Efficiency  (%)", fontsize=12)
ax_q.set_xlim(0, THR_Q_MAX); ax_q.set_yscale("log"); ax_q.set_ylim(0.01, 110)
ax_q.axhline(100, color="lightgray", lw=0.8, ls="--")
ax_q.legend(fontsize=10); ax_q.grid(True, alpha=0.3, which="both")

# Asse superiore in SPE
ax_q2 = ax_q.twiny()
ax_q2.set_xlim(0, 10.0)
ax_q2.set_xlabel("Threshold  (SPE)", fontsize=11)
ax_q2.tick_params(axis="x")

plt.tight_layout()
fig_q.savefig(os.path.join(OUTPUT_DIR, "efficiency_vs_threshold_charge.pdf"), bbox_inches="tight")
plt.show()

# Stampa valori a soglie significative (picco)
print(f"\n--- Efficienza vs soglia picco ---")
print(f"\n{'|Thr| (mV)':>12} | {'SPE':>6}", end="")
for src_name in results: print(f" | {'Eff '+src_name+' (%)':>16}", end="")
print()
print("-" * (12 + 8 + 18*len(results)))
for thr_val in [0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 6.0, 8.0, 10.0]:
    spe_label = f"{thr_val/abs(amp_SPE_mV):.1f}"
    print(f"{thr_val:>12.1f} | {spe_label:>6}", end="")
    for src_name, res in results.items():
        pmin = np.minimum(res["peak_a"], res["peak_b"])
        eff  = 100.0 * (pmin < -thr_val).sum() / res["n_hits"]
        print(f" | {eff:>16.2f}", end="")
    print()

# Stampa valori a soglie significative (carica)
print(f"\n--- Efficienza vs soglia carica ---")
print(f"\n{'ThrQ (ADC)':>12} | {'SPE':>6}", end="")
for src_name in results: print(f" | {'Eff '+src_name+' (%)':>16}", end="")
print()
print("-" * (12 + 8 + 18*len(results)))
for k in [0, 0.5, 1, 2, 3, 5, 7, 10]:
    thr_val = k * q_SPE
    print(f"{thr_val:>12.1f} | {k:>6.1f}", end="")
    for src_name, res in results.items():
        q_mx = np.maximum(res["qa"], res["qb"])
        eff  = 100.0 * (q_mx > thr_val).sum() / res["n_hits"]
        print(f" | {eff:>16.2f}", end="")
    print()

print("\nStatistiche fotoni al PMT (eventi con ≥1 fotone):")
print(f"{'Sorgente':12} | {'media':>8} | {'mediana':>8} | {'max':>6}")
print("-"*44)
for src, res in results.items():
    arr = res["n_ph_pmt"]
    print(f"{src:12} | {arr.mean():>8.1f} | {np.median(arr):>8.1f} | {arr.max():>6d}")
print("="*65)

# ============================================================
# === EFFICIENZA DI MODULO  (A∪B, logica picco + carica) =====
#   Denominatore: n_hits = eventi con ≥1 fotone al PMT
# ============================================================
print(f"\n{'='*70}")
print(f"EFFICIENZA DI MODULO  (A∪B)  —  denom = N eventi con fotoni al PMT")
print(f"  Condizione picco:   peak_A < {THR_PEAK_MV:.0f} mV  OR  peak_B < {THR_PEAK_MV:.0f} mV")
print(f"  Condizione carica:  max(QA, QB) > {THR_CHARGE:.0f} ADC·samples")
print(f"{'='*70}")
print(f"\n{'Sorgente':12} | {'N hits':>8} | {'N picco':>10} | {'N carica':>10} | "
      f"{'N P∩Q':>8} | {'Eff P (%)':>10} | {'Eff Q (%)':>10} | {'Eff P∩Q (%)':>12}")
print("-"*98)

_double_results = {}
for src_name, res in results.items():
    peak_min = np.minimum(res["peak_a"], res["peak_b"])
    q_mx     = np.maximum(res["qa"],    res["qb"])
    n_denom  = res["n_hits"]
    mask_p   = peak_min < THR_PEAK_MV
    mask_q   = q_mx     > THR_CHARGE
    mask_d   = mask_p & mask_q
    n_p, n_q, n_d = mask_p.sum(), mask_q.sum(), mask_d.sum()
    eff_p = 100. * n_p / n_denom
    eff_q = 100. * n_q / n_denom
    eff_d = 100. * n_d / n_denom
    _double_results[src_name] = {"eff_p": eff_p, "eff_q": eff_q, "eff_d": eff_d,
                                  "n_denom": n_denom, "n_d": n_d}
    print(f"{src_name:12} | {n_denom:>8d} | {n_p:>10d} | {n_q:>10d} | "
          f"{n_d:>8d} | {eff_p:>9.2f}% | {eff_q:>9.2f}% | {eff_d:>11.2f}%")
print("="*98)

# --- Plot picco per modulo: min(peak_A, peak_B) ---
fig8, ax8 = plt.subplots(figsize=(9, 6))
fig8.suptitle(
    f"Module peak amplitude = min(peak_A, peak_B)\n"
    f"gate [{GATE_START_NS:.0f}–{GATE_END_NS:.0f} ns]  —  Thr = {THR_PEAK_MV:.0f} mV",
    fontsize=12)
_all_pmin = np.concatenate([np.minimum(r["peak_a"], r["peak_b"]) for r in results.values()])
_pbins = np.linspace(np.percentile(_all_pmin, 0.5), np.percentile(_all_pmin, 99.5), 80)
for src_name, res in results.items():
    pmin = np.minimum(res["peak_a"], res["peak_b"])
    ax8.hist(pmin, bins=_pbins, density=True,
             color=res["cfg"]["color"], alpha=0.55, histtype="stepfilled",
             edgecolor=res["cfg"]["color"], linewidth=1.1, label=res["cfg"]["label"])
ax8.axvline(THR_PEAK_MV, color="tomato", lw=2.0, ls="--",
            label=f"Thr = {THR_PEAK_MV:.0f} mV")
ax8.set_xlabel("min(peak_A, peak_B)  —  peak of the most active channel (mV)", fontsize=11)
ax8.set_ylabel("Probability density", fontsize=11)
ax8.set_yscale("log")
ax8.legend(fontsize=10); ax8.grid(True, alpha=0.3, which="both")
plt.tight_layout()
fig8.savefig(os.path.join(OUTPUT_DIR, "peak_amplitude_dist_gamma_vs_mu_lngs.pdf"), bbox_inches="tight")
plt.show()

print(f"\n{'Sorgente':12} | {'Eff picco (%)':>14} | {'Eff carica (%)':>15} | {'Eff P∩Q (%)':>12}")
print("-"*58)
for src_name, dr in _double_results.items():
    print(f"{src_name:12} | {dr['eff_p']:>13.2f}% | {dr['eff_q']:>14.2f}% | {dr['eff_d']:>11.2f}%")
print("="*58)

# ============================================================
# === SCATTER: peak_A vs peak_B  (gamma e mu_lngs) ===========
# ============================================================
n_src = len(results)
fig9, axes9 = plt.subplots(1, n_src, figsize=(7 * n_src, 6), squeeze=False)
fig9.suptitle(
    f"Peak amplitude:  PMT A vs PMT B  —  gate [{GATE_START_NS:.0f}–{GATE_END_NS:.0f} ns]",
    fontsize=13)

for ax9, (src_name, res) in zip(axes9[0], results.items()):
    pa = res["peak_a"]; pb = res["peak_b"]
    col = res["cfg"]["color"]; lbl = res["cfg"]["label"]
    ax9.scatter(pa, pb, s=3, alpha=0.12, color=col, linewidths=0, label=f"{lbl}  (N={len(pa)})")
    _lo = min(pa.min(), pb.min()); _hi = max(pa.max(), pb.max())
    ax9.plot([_lo, _hi], [_lo, _hi], color="gray", lw=0.9, ls=":", label="A = B")
    ax9.set_xlabel("peak_A  (mV)", fontsize=12)
    ax9.set_ylabel("peak_B  (mV)", fontsize=12)
    ax9.set_title(lbl, fontsize=12)
    ax9.legend(fontsize=9, markerscale=4)
    ax9.grid(True, alpha=0.3)

plt.tight_layout()
fig9.savefig(os.path.join(OUTPUT_DIR, "scatter_peakA_vs_peakB.pdf"), bbox_inches="tight")
plt.show()

# --- versione hexbin (densità 2D, scala log) ---
fig10, axes10 = plt.subplots(1, n_src, figsize=(7 * n_src, 6), squeeze=False)
fig10.suptitle(
    f"2D density  peak_A vs peak_B  (hexbin, log)  —  gate [{GATE_START_NS:.0f}–{GATE_END_NS:.0f} ns]",
    fontsize=12)

for ax10, (src_name, res) in zip(axes10[0], results.items()):
    pa = res["peak_a"]; pb = res["peak_b"]
    lbl = res["cfg"]["label"]
    hb = ax10.hexbin(pa, pb, gridsize=60, bins="log", cmap="viridis", linewidths=0.2)
    plt.colorbar(hb, ax=ax10, label="log₁₀(counts)")
    _lo = min(pa.min(), pb.min()); _hi = max(pa.max(), pb.max())
    ax10.plot([_lo, _hi], [_lo, _hi], color="white", lw=0.9, ls=":", alpha=0.7, label="A = B")
    ax10.set_xlabel("peak_A  (mV)", fontsize=12)
    ax10.set_ylabel("peak_B  (mV)", fontsize=12)
    ax10.set_title(lbl, fontsize=12)
    ax10.legend(fontsize=9)
    ax10.grid(True, alpha=0.2, color="white")

plt.tight_layout()
fig10.savefig(os.path.join(OUTPUT_DIR, "hexbin_peakA_vs_peakB.pdf"), bbox_inches="tight")
plt.show()

# ============================================================
# === CONTEGGIO SEGNALE A / B / A∩B  (solo gamma) ============
# ============================================================
src = "gamma"
if src in results:
    pa = results[src]["peak_a"]
    pb = results[src]["peak_b"]
    n_hits_g = results[src]["n_hits"]

    print("=== DIAGNOSTICA PICCHI GAMMA ===")
    print(f"  peak_A:  min={pa.min():.3f} mV  max={pa.max():.3f} mV  media={pa.mean():.3f} mV")
    print(f"  peak_B:  min={pb.min():.3f} mV  max={pb.max():.3f} mV  media={pb.mean():.3f} mV")
    print(f"  amp_SPE = {amp_SPE_mV:.2f} mV  →  Thr {THR_PEAK_MV:.0f} mV ≈ {abs(THR_PEAK_MV/amp_SPE_mV):.1f} SPE")
    print()

    sig_A = pa < THR_PEAK_MV
    sig_B = pb < THR_PEAK_MV
    n_only_A = ( sig_A & ~sig_B).sum()
    n_only_B = (~sig_A &  sig_B).sum()
    n_both   = ( sig_A &  sig_B).sum()
    n_any    = n_only_A + n_only_B + n_both

    print(f"=== CONTEGGIO  (soglia = {THR_PEAK_MV:.0f} mV ≈ {abs(THR_PEAK_MV/amp_SPE_mV):.1f} SPE) ===")
    print(f"  N eventi con fotoni al PMT: {len(pa)}")
    if n_any > 0:
        print(f"\n  --- eventi con segnale (N={n_any}) ---")
        print(f"  Solo A   (A=1, B=0): {n_only_A:>5d}  ({100*n_only_A/n_any:5.1f}%)")
        print(f"  Solo B   (A=0, B=1): {n_only_B:>5d}  ({100*n_only_B/n_any:5.1f}%)")
        print(f"  Entrambi (A=1, B=1): {n_both:>5d}  ({100*n_both/n_any:5.1f}%)")

    fig_d, (axd_a, axd_b) = plt.subplots(1, 2, figsize=(14, 5))
    fig_d.suptitle(f"Gamma — peak distribution in gate [{GATE_START_NS:.0f}–{GATE_END_NS:.0f} ns]", fontsize=12)
    for ax_d, arr, ch_lbl in [(axd_a, pa, "peak_A"), (axd_b, pb, "peak_B")]:
        ax_d.hist(arr, bins=60, color="steelblue", alpha=0.7, edgecolor="steelblue", linewidth=0.8)
        ax_d.axvline(THR_PEAK_MV, color="tomato", lw=2, ls="--", label=f"Thr = {THR_PEAK_MV:.0f} mV")
        for k in [1, 2, 3]:
            ax_d.axvline(k * amp_SPE_mV, color="green", lw=1, ls=":",
                         label=f"{k} SPE = {k*amp_SPE_mV:.2f} mV" if k == 1 else f"{k} SPE")
        ax_d.set_xlabel(f"{ch_lbl}  (mV)", fontsize=11)
        ax_d.set_ylabel("Counts", fontsize=11)
        ax_d.set_title(ch_lbl, fontsize=11)
        ax_d.legend(fontsize=9); ax_d.grid(True, alpha=0.3)
    plt.tight_layout()
    fig_d.savefig(os.path.join(OUTPUT_DIR, "gamma_peak_diagnostic.pdf"), bbox_inches="tight")
    plt.show()

# ============================================================
# === DIAGNOSTICA: fotoni rivelati per canale (gamma) =========
# ============================================================
if "gamma" in results:
    fpath = os.path.join(DATA_DIR, results["gamma"]["cfg"]["file"])
    with uproot.open(fpath) as f:
        df_g = f["Fotoni"].arrays(
            ["EventID","Arrival_Time_ns","PMT_ID","E_Hit_eV"], library="pd")

    ev_ids_g = np.sort(df_g["EventID"].unique())
    n_ph_A_raw, n_ph_B_raw   = [], []
    n_ph_A_det, n_ph_B_det   = [], []
    n_ph_A_gate, n_ph_B_gate = [], []

    rng2 = np.random.default_rng(0)
    for ev_id in ev_ids_g:
        hits = df_g[df_g["EventID"] == ev_id]
        for ch, raw_list, det_list, gate_list in [
                (CH_A, n_ph_A_raw, n_ph_A_det, n_ph_A_gate),
                (CH_B, n_ph_B_raw, n_ph_B_det, n_ph_B_gate)]:
            ph_ch = hits[hits["PMT_ID"] == ch]
            n_raw = len(ph_ch)
            raw_list.append(n_raw)
            if n_raw == 0:
                det_list.append(0); gate_list.append(0); continue
            wl_nm  = 1240.0 / ph_ch["E_Hit_eV"].values
            mask   = rng2.random(n_raw) < get_qe(wl_nm)
            n_det  = mask.sum()
            det_list.append(n_det)
            t_acc  = ph_ch["Arrival_Time_ns"].values[mask] + TRIGGER_OFFSET_NS
            n_gate = ((t_acc >= GATE_START_NS) & (t_acc < GATE_END_NS)).sum()
            gate_list.append(int(n_gate))

    n_ph_A_raw  = np.array(n_ph_A_raw);  n_ph_B_raw  = np.array(n_ph_B_raw)
    n_ph_A_det  = np.array(n_ph_A_det);  n_ph_B_det  = np.array(n_ph_B_det)
    n_ph_A_gate = np.array(n_ph_A_gate); n_ph_B_gate = np.array(n_ph_B_gate)

    print("=== FOTONI PER EVENTO — gamma ===")
    print(f"{'':30} {'ch A':>10} {'ch B':>10}")
    print(f"  Media fotoni Geant4 (pre-QE):  {n_ph_A_raw.mean():>10.2f} {n_ph_B_raw.mean():>10.2f}")
    print(f"  Media fotoni rivelati (post-QE):{n_ph_A_det.mean():>10.2f} {n_ph_B_det.mean():>10.2f}")
    print(f"  Media fotoni nel gate:          {n_ph_A_gate.mean():>10.2f} {n_ph_B_gate.mean():>10.2f}")
    print()
    for ch_lbl, arr in [("ch A", n_ph_A_det), ("ch B", n_ph_B_det)]:
        print(f"  Distribuzione fotoni rivelati ({ch_lbl}):")
        vals, cnts = np.unique(arr, return_counts=True)
        for v, c in zip(vals, cnts):
            print(f"    {v:2d} fotoni → {c:4d} eventi  ({100*c/len(arr):.1f}%)")
    print()
    n_both_zero = ((n_ph_A_gate == 0) & (n_ph_B_gate == 0)).sum()
    print(f"  Eventi con 0 fotoni nel gate su entrambi i canali: {n_both_zero} / {len(ev_ids_g)}")
    print(f"  (questi danno picco = 0 esatto con NOISE_RMS_ADC = 0)")


In [ ]:
# ============================================================
# PLOT COMBINATO: efficienza vs soglia carica
#   con soglia di picco FISSA a -1 mV
#
#   Condizione A∪B:
#     min(peak_A, peak_B) < PEAK_FIXED_MV   (picco fisso)
#     AND
#     max(QA, QB) > ThrQ                    (carica variabile)
#
#   X-axis: ThrQ da 0 a 10 SPE (in ADC·samples)
#   Asse superiore in SPE
#   Denominatore: n_hits
# ============================================================

PEAK_FIXED_MV   = -1.0          # soglia di picco fissa
THR_Q_MAX_COMB  = 10.0 * q_SPE  # 10 SPE in ADC·samples
THR_Q_COMB      = np.linspace(0, THR_Q_MAX_COMB, 500)

fig_comb, ax_comb = plt.subplots(figsize=(9, 6))
fig_comb.suptitle(
    rf"Combined efficiency  (A∪B)"
    "\n"
    rf"fixed peak $<$ {abs(PEAK_FIXED_MV):.0f} mV  AND  max(QA,QB) $>$ ThrQ  —  denom = $N_{{hits}}$",
    fontsize=12)

for src_name, res in results.items():
    pmin       = np.minimum(res["peak_a"], res["peak_b"])
    q_mx       = np.maximum(res["qa"],     res["qb"])
    n_hits_src = res["n_hits"]
    mask_peak  = pmin < PEAK_FIXED_MV          # picco fisso a -1 mV

    effs_comb = np.array([
        100.0 * (mask_peak & (q_mx > thr)).sum() / n_hits_src
        for thr in THR_Q_COMB
    ])
    # solo carica (per confronto, tratteggiata)
    effs_q = np.array([
        100.0 * (q_mx > thr).sum() / n_hits_src
        for thr in THR_Q_COMB
    ])

    col = res["cfg"]["color"]
    lbl = res["cfg"]["label"]
    ax_comb.plot(THR_Q_COMB, effs_comb, color=col, lw=2.2,
                 label=f"{lbl}  (P∩Q)")
    ax_comb.plot(THR_Q_COMB, effs_q,    color=col, lw=1.1, ls="--", alpha=0.6,
                 label=f"{lbl}  (Q only, ref.)")

# Marcatori SPE in verde
for k in [1, 2, 3, 5, 10]:
    q_k = k * q_SPE
    if q_k <= THR_Q_MAX_COMB:
        ax_comb.axvline(q_k, color="green", lw=1.0, ls=":",
                        label=(f"1 SPE = {q_k:.0f} ADC·samples" if k == 1 else f"{k} SPE"))

# Soglia nominale carica
ax_comb.axvline(THR_CHARGE, color="gray", lw=1.2, ls="--",
                label=f"Nominal ThrQ = {THR_CHARGE:.0f} ADC·samples")

ax_comb.set_xlabel(
    r"ThrQ  (ADC·samples)  —  signal if peak $<$ −1 mV  AND  max(QA,QB) $>$ ThrQ",
    fontsize=11)
ax_comb.set_ylabel(r"Efficiency  (%)", fontsize=12)
ax_comb.set_xlim(0, THR_Q_MAX_COMB)
ax_comb.set_yscale("log"); ax_comb.set_ylim(0.01, 110)
ax_comb.axhline(100, color="lightgray", lw=0.8, ls="--")
ax_comb.legend(fontsize=9, ncol=2)
ax_comb.grid(True, alpha=0.3, which="both")

# Asse superiore in SPE
ax_comb2 = ax_comb.twiny()
ax_comb2.set_xlim(0, 10.0)
ax_comb2.set_xlabel("Charge threshold  (SPE)", fontsize=11)

plt.tight_layout()
fig_comb.savefig(
    os.path.join(OUTPUT_DIR, "efficiency_combined_peak1mV_vs_charge.pdf"),
    bbox_inches="tight")
plt.show()

# --- Tabella numerica ---
print(f"\n--- Eff combinata: picco < {PEAK_FIXED_MV:.0f} mV  AND  Q > ThrQ ---")
print(f"\n{'ThrQ (ADC)':>12} | {'SPE':>6}", end="")
for src_name in results:
    print(f" | {'P∩Q '+src_name+' (%)':>18}", end="")
print()
print("-" * (12 + 8 + 20 * len(results)))
for k in [0, 0.5, 1, 2, 3, 5, 7, 10]:
    thr_val = k * q_SPE
    print(f"{thr_val:>12.1f} | {k:>6.1f}", end="")
    for src_name, res in results.items():
        pmin = np.minimum(res["peak_a"], res["peak_b"])
        q_mx = np.maximum(res["qa"],     res["qb"])
        eff  = 100.0 * ((pmin < PEAK_FIXED_MV) & (q_mx > thr_val)).sum() / res["n_hits"]
        print(f" | {eff:>18.2f}", end="")
    print()


In [ ]:
# ============================================================
# STUDIO DI OTTIMIZZAZIONE — scan 2D delle soglie
#
#   Segnale: mu_lngs     Fondo: gamma
#   Griglia: N_PEAK × N_CHG punti in (|thr_peak|, ThrQ)
#   Output:
#     - Heatmap 2D: eff_mu, eff_gamma, FOM
#     - Curva ROC con frontiera di Pareto
#     - Tabella Pareto con rate fisiche
# ============================================================

SRC_SIG = "mu_lngs"
SRC_BKG = "gamma"

# --- Rate fisiche di riferimento ---
# Muoni: flusso Mei & Hime (LNGS Hall A, 3800 m.w.e.), piano 6500×6500 mm
PHI_MU_CM2_S  = 3.0e-8               # cm⁻²s⁻¹  (Mei & Hime 2006)
A_GEN_CM2     = (6500e-1) ** 2        # 650 cm × 650 cm = 422500 cm²
RATE_MU_GEN_HZ = PHI_MU_CM2_S * A_GEN_CM2   # Hz muoni generati sul piano

# Gamma ambientali — normalizzazione ASSOLUTA dalla stessa misura usata
# per la forma dello spettro (arXiv:1101.5298): flusso integrale Hall A
# (E < 3 MeV): φ = 0.3–0.4 cm⁻²s⁻¹. La sorgente MC genera con legge del
# coseno dalla superficie del parallelepipedo 1300×1300×300 mm: per un
# campo isotropo la rate entrante è R = (φ/4)·A_gen.
PHI_GAMMA_CM2S  = 0.35                               # cm⁻²s⁻¹ (0.3–0.4)
A_GEN_GAMMA_CM2 = 2*(130.0*130.0) + 4*(130.0*30.0)   # = 49400 cm²
RATE_GAMMA_HZ   = PHI_GAMMA_CM2S/4.0 * A_GEN_GAMMA_CM2   # ≈ 4.3 kHz

SEC_PER_DAY = 86400.0

if SRC_SIG not in results or SRC_BKG not in results:
    print(f"Sorgenti richieste: {SRC_SIG}, {SRC_BKG}. Controllare che i file ROOT esistano.")
else:
    # --- Array per segnale e fondo ---
    pmin_sig = np.minimum(results[SRC_SIG]["peak_a"], results[SRC_SIG]["peak_b"])
    q_mx_sig = np.maximum(results[SRC_SIG]["qa"],     results[SRC_SIG]["qb"])
    n_sig    = results[SRC_SIG]["n_hits"]
    n_gen_mu = results[SRC_SIG]["n_gen"]

    pmin_bkg = np.minimum(results[SRC_BKG]["peak_a"], results[SRC_BKG]["peak_b"])
    q_mx_bkg = np.maximum(results[SRC_BKG]["qa"],     results[SRC_BKG]["qb"])
    n_bkg    = results[SRC_BKG]["n_hits"]
    n_gen_gam = results[SRC_BKG]["n_gen"]

    # Accettanza × efficienza ottica complessiva (da simulazione MC)
    acc_mu  = n_sig  / n_gen_mu   # P(fotoni al PMT | muone generato sul piano)
    acc_gam = n_bkg  / n_gen_gam  # P(fotoni al PMT | gamma generato)

    # Rate base (a soglia = 0, solo condizionata all'aver fotoni al PMT)
    R0_mu_day  = RATE_MU_GEN_HZ  * acc_mu  * SEC_PER_DAY   # eventi/giorno a ε_mu=100%
    R0_gam_day = RATE_GAMMA_HZ   * acc_gam * SEC_PER_DAY   # eventi/giorno a ε_gam=100%

    print(f"Rate muoni generati sul piano:   {RATE_MU_GEN_HZ*1e3:.2f} mHz  "
          f"({RATE_MU_GEN_HZ*SEC_PER_DAY:.1f} ev/giorno)")
    print(f"Accettanza ottica muoni (MC):    {acc_mu*100:.3f}%")
    print(f"Rate muoni con fotoni al PMT:    {R0_mu_day:.3f} ev/giorno")
    print()
    print(f"Rate gamma sul rivelatore:       {RATE_GAMMA_HZ:.1f} Hz  "
          f"({RATE_GAMMA_HZ*SEC_PER_DAY:.0f} ev/giorno)")
    print(f"Accettanza ottica gamma (MC):    {acc_gam*100:.3f}%")
    print(f"Rate gamma con fotoni al PMT:    {R0_gam_day:.1f} ev/giorno")

    # --- Griglia delle soglie ---
    N_PEAK = 60
    N_CHG  = 60
    thr_p_vals = np.linspace(0.2, 10.0,          N_PEAK)  # |thr_peak| in mV
    thr_q_vals = np.linspace(0,   10.0 * q_SPE,  N_CHG)   # ThrQ in ADC·samples

    thr_p_spe = thr_p_vals / abs(amp_SPE_mV)
    thr_q_spe = thr_q_vals / q_SPE

    # --- Calcolo matriciale (broadcasting) ---
    mask_p_sig = (pmin_sig[:, None] < -thr_p_vals[None, :]).astype(np.float32)
    mask_q_sig = (q_mx_sig[:, None] >  thr_q_vals[None, :]).astype(np.float32)
    mask_p_bkg = (pmin_bkg[:, None] < -thr_p_vals[None, :]).astype(np.float32)
    mask_q_bkg = (q_mx_bkg[:, None] >  thr_q_vals[None, :]).astype(np.float32)

    EFF_SIG = 100.0 * (mask_p_sig.T @ mask_q_sig) / n_sig   # (N_PEAK, N_CHG)
    EFF_BKG = 100.0 * (mask_p_bkg.T @ mask_q_bkg) / n_bkg

    # Rate rivelate sulla griglia (eventi/giorno)
    RATE_MU_GRID  = R0_mu_day  * EFF_SIG / 100.0
    RATE_GAM_GRID = R0_gam_day * EFF_BKG / 100.0

    print(f"\nGriglia: {N_PEAK}×{N_CHG} = {N_PEAK*N_CHG} punti calcolati")

    # --- Figure of Merit ---
    EPS_BKG = 0.1
    FOM_R   = EFF_SIG / (EFF_BKG + EPS_BKG)
    FOM_D   = EFF_SIG - EFF_BKG
    FOM_S   = EFF_SIG / (np.sqrt(EFF_BKG + EPS_BKG))

    idx_R = np.unravel_index(FOM_R.argmax(), FOM_R.shape)
    idx_D = np.unravel_index(FOM_D.argmax(), FOM_D.shape)
    idx_S = np.unravel_index(FOM_S.argmax(), FOM_S.shape)

    print("\n=== PUNTI OTTIMALI ===")
    for label, idx, fom in [
        (r"ε_μ / ε_γ",   idx_R, FOM_R),
        (r"ε_μ − ε_γ",   idx_D, FOM_D),
        (r"ε_μ / √ε_γ",  idx_S, FOM_S),
    ]:
        ip, iq = idx
        print(f"  FOM = {label}:")
        print(f"    |thr_peak| = {thr_p_vals[ip]:.2f} mV ({thr_p_spe[ip]:.2f} SPE)"
              f",  ThrQ = {thr_q_vals[iq]:.1f} ADC·samples ({thr_q_spe[iq]:.2f} SPE)")
        print(f"    eff_mu = {EFF_SIG[ip,iq]:.1f}%,  eff_gamma = {EFF_BKG[ip,iq]:.2f}%"
              f",  FOM = {fom[ip,iq]:.2f}")
        print(f"    R_mu = {RATE_MU_GRID[ip,iq]:.4f} ev/giorno"
              f",  R_gamma = {RATE_GAM_GRID[ip,iq]:.2f} ev/giorno")

    # === PLOT 1: heatmap eff_mu, eff_gamma, FOM_R ===
    Q_mesh, P_mesh = np.meshgrid(thr_q_spe, thr_p_spe)

    fig_h, axes_h = plt.subplots(1, 3, figsize=(18, 6))
    fig_h.suptitle(
        r"2D threshold scan — signal: $\mu^-$ LNGS,  background: $\gamma$  (A∪B, P∩Q)",
        fontsize=13)

    for ax, data, title, cmap, idx_opt in [
        (axes_h[0], EFF_SIG, r"Signal eff. $\epsilon_\mu$  (%)",  "Blues",   None),
        (axes_h[1], EFF_BKG, r"Background eff. $\epsilon_\gamma$  (%)", "Reds",    None),
        (axes_h[2], FOM_R,   r"FOM = $\epsilon_\mu\,/\,\epsilon_\gamma$", "viridis", idx_R),
    ]:
        im = ax.pcolormesh(Q_mesh, P_mesh, data, cmap=cmap, shading="auto")
        plt.colorbar(im, ax=ax, shrink=0.85)
        ax.set_xlabel("ThrQ  (SPE)", fontsize=11)
        ax.set_ylabel("|peak threshold|  (SPE)", fontsize=11)
        ax.set_title(title, fontsize=11)
        ax.axhline(abs(THR_PEAK_MV) / abs(amp_SPE_mV), color="white", lw=1.0, ls="--", alpha=0.7)
        ax.axvline(THR_CHARGE / q_SPE,                  color="white", lw=1.0, ls="--", alpha=0.7)
        if idx_opt is not None:
            ip, iq = idx_opt
            ax.plot(thr_q_spe[iq], thr_p_spe[ip], "w*", ms=14,
                    label=f"optimum\n({thr_p_vals[ip]:.1f} mV, {thr_q_spe[iq]:.1f} SPE)")
            ax.legend(fontsize=9)

    plt.tight_layout()
    fig_h.savefig(os.path.join(OUTPUT_DIR, "optimization_2D_heatmap.pdf"), bbox_inches="tight")
    plt.show()

    # === PLOT 2: heatmap rate rivelate ===
    fig_hr, axes_hr = plt.subplots(1, 2, figsize=(14, 6))
    fig_hr.suptitle(
        f"Detected rates (ev/day)  —  "
        f"Φ_μ = {PHI_MU_CM2_S:.0e} cm⁻²s⁻¹,  R_γ = {RATE_GAMMA_HZ:.0f} Hz",
        fontsize=12)

    for ax, data, title, cmap in [
        (axes_hr[0], RATE_MU_GRID,  r"Detected $\mu$ rate  (ev/day)", "Blues"),
        (axes_hr[1], RATE_GAM_GRID, r"Detected $\gamma$ rate  (ev/day)", "Reds"),
    ]:
        im = ax.pcolormesh(Q_mesh, P_mesh, data, cmap=cmap, shading="auto")
        plt.colorbar(im, ax=ax, shrink=0.85)
        ax.set_xlabel("ThrQ  (SPE)", fontsize=11)
        ax.set_ylabel("|peak threshold|  (SPE)", fontsize=11)
        ax.set_title(title, fontsize=11)
        ax.axhline(abs(THR_PEAK_MV) / abs(amp_SPE_mV), color="white", lw=1.0, ls="--", alpha=0.7)
        ax.axvline(THR_CHARGE / q_SPE,                  color="white", lw=1.0, ls="--", alpha=0.7)

    plt.tight_layout()
    fig_hr.savefig(os.path.join(OUTPUT_DIR, "optimization_2D_rates.pdf"), bbox_inches="tight")
    plt.show()

    # === PLOT 3: heatmap FOM_D e FOM_S ===
    fig_h2, axes_h2 = plt.subplots(1, 2, figsize=(14, 6))
    fig_h2.suptitle("Alternative FOMs for threshold optimisation", fontsize=13)

    for ax, data, title, idx_opt in [
        (axes_h2[0], FOM_D, r"FOM = $\epsilon_\mu - \epsilon_\gamma$",         idx_D),
        (axes_h2[1], FOM_S, r"FOM = $\epsilon_\mu\,/\,\sqrt{\epsilon_\gamma}$", idx_S),
    ]:
        im = ax.pcolormesh(Q_mesh, P_mesh, data, cmap="viridis", shading="auto")
        plt.colorbar(im, ax=ax, shrink=0.85)
        ax.set_xlabel("ThrQ  (SPE)", fontsize=11)
        ax.set_ylabel("|peak threshold|  (SPE)", fontsize=11)
        ax.set_title(title, fontsize=11)
        ax.axhline(abs(THR_PEAK_MV) / abs(amp_SPE_mV), color="white", lw=1.0, ls="--", alpha=0.7)
        ax.axvline(THR_CHARGE / q_SPE,                  color="white", lw=1.0, ls="--", alpha=0.7)
        ip, iq = idx_opt
        ax.plot(thr_q_spe[iq], thr_p_spe[ip], "w*", ms=14,
                label=f"optimum\n({thr_p_vals[ip]:.1f} mV, {thr_q_spe[iq]:.1f} SPE)")
        ax.legend(fontsize=9)

    plt.tight_layout()
    fig_h2.savefig(os.path.join(OUTPUT_DIR, "optimization_2D_altFOM.pdf"), bbox_inches="tight")
    plt.show()

    # === PLOT 4: curva ROC + frontiera di Pareto ===
    eff_s_flat = EFF_SIG.ravel()
    eff_b_flat = EFF_BKG.ravel()

    idx_sort    = np.argsort(eff_b_flat)
    g_sort      = eff_b_flat[idx_sort]
    m_sort      = eff_s_flat[idx_sort]
    pareto_mask = np.zeros(len(g_sort), dtype=bool)
    run_max     = -np.inf
    for k in range(len(g_sort)):
        if m_sort[k] > run_max:
            pareto_mask[k] = True
            run_max = m_sort[k]

    fom_flat = FOM_R.ravel()
    vmax_fom = np.percentile(fom_flat, 98)

    fig_roc, ax_roc = plt.subplots(figsize=(8, 7))
    fig_roc.suptitle(
        r"ROC curve — 2D threshold scan (peak AND charge)"
        "\n"
        r"Signal: $\mu^-$ LNGS   Background: $\gamma$ LNGS",
        fontsize=12)

    sc = ax_roc.scatter(eff_b_flat, eff_s_flat,
                        c=fom_flat, cmap="plasma",
                        vmin=0, vmax=vmax_fom,
                        s=14, alpha=0.45, linewidths=0, zorder=2)
    plt.colorbar(sc, ax=ax_roc,
                 label=r"FOM = $\epsilon_\mu\,/\,\epsilon_\gamma$  (saturated at 98th perc.)",
                 shrink=0.85)

    p_g = g_sort[pareto_mask]; p_m = m_sort[pareto_mask]
    srt2 = np.argsort(p_g)
    ax_roc.plot(p_g[srt2], p_m[srt2], "k-", lw=2.2, zorder=4, label="Pareto frontier")
    ax_roc.plot(p_g[srt2], p_m[srt2], "ko", ms=5,  zorder=4)

    ip_c = np.argmin(np.abs(thr_p_vals - abs(THR_PEAK_MV)))
    iq_c = np.argmin(np.abs(thr_q_vals - THR_CHARGE))
    ax_roc.plot(EFF_BKG[ip_c, iq_c], EFF_SIG[ip_c, iq_c],
                "rs", ms=11, zorder=5,
                label=f"Current thresholds ({abs(THR_PEAK_MV):.0f} mV, {THR_CHARGE:.0f} ADC·samples)")

    ip_R, iq_R = idx_R
    ax_roc.plot(EFF_BKG[ip_R, iq_R], EFF_SIG[ip_R, iq_R],
                "w*", ms=16, markeredgecolor="k", markeredgewidth=0.8, zorder=5,
                label=(f"Optimum ε/ε: {thr_p_vals[ip_R]:.1f} mV, "
                       f"{thr_q_spe[iq_R]:.1f} SPE  "
                       f"→ ε_μ={EFF_SIG[ip_R,iq_R]:.0f}%, ε_γ={EFF_BKG[ip_R,iq_R]:.1f}%"))

    ax_roc.plot([0, 40], [0, 40], color="green", lw=0.9, ls=":", zorder=1,
                label="Random classifier")

    ax_roc.set_xlabel(r"Background efficiency $\epsilon_\gamma$  (%)", fontsize=12)
    ax_roc.set_ylabel(r"Signal efficiency $\epsilon_\mu$  (%)", fontsize=12)
    ax_roc.set_xlim(-0.5, 40); ax_roc.set_ylim(-1, 101)
    ax_roc.grid(True, alpha=0.3)
    ax_roc.legend(fontsize=9, loc="lower right")

    plt.tight_layout()
    fig_roc.savefig(os.path.join(OUTPUT_DIR, "optimization_ROC.pdf"), bbox_inches="tight")
    plt.show()

    # === Tabella frontiera di Pareto con rate fisiche ===
    print(f"\n--- Pareto frontier (campionata)  ---")
    print(f"    Φ_μ = {PHI_MU_CM2_S:.0e} cm⁻²s⁻¹,  A_gen = {A_GEN_CM2:.0f} cm²"
          f"  →  R_μ_gen = {RATE_MU_GEN_HZ*1e3:.2f} mHz")
    print(f"    R_γ (superficie rivelatore) = {RATE_GAMMA_HZ:.1f} Hz  ← configura RATE_GAMMA_HZ")
    print()

    hdr = (f"{'ε_γ (%)':>8} | {'ε_μ (%)':>8} | {'FOM ε/ε':>8} | "
           f"{'|thr_pk|':>9} | {'ThrQ':>6} | "
           f"{'R_μ (ev/d)':>12} | {'R_γ (ev/d)':>12} | {'R_μ/R_γ':>10}")
    print(hdr)
    print("-" * len(hdr))

    n_pareto = pareto_mask.sum()
    step = max(1, n_pareto // 12)
    idx_samp = np.where(pareto_mask)[0][::step]

    for k in idx_samp:
        g_k  = g_sort[k]; m_k = m_sort[k]
        fom_k = m_k / (g_k + EPS_BKG)
        orig_k = idx_sort[k]
        ip_k, iq_k = np.unravel_index(orig_k, EFF_SIG.shape)
        r_mu_k  = R0_mu_day  * m_k / 100.0
        r_gam_k = R0_gam_day * g_k / 100.0
        ratio_k = r_mu_k / r_gam_k if r_gam_k > 0 else float("inf")
        print(f"{g_k:>8.2f} | {m_k:>8.1f} | {fom_k:>8.2f} | "
              f"{thr_p_vals[ip_k]:>7.2f}mV | {thr_q_spe[iq_k]:>4.1f}SPE | "
              f"{r_mu_k:>12.4f} | {r_gam_k:>12.2f} | {ratio_k:>10.4f}")


In [ ]:
# ============================================================
# CONFRONTO  single  vs  single_wbc2
#
#   Per ogni sorgente (mu_lngs, gamma) digitizza i ROOT WBC2
#   con gli stessi parametri del loop principale e confronta:
#     1. N fotoni Cherenkov prodotti e entrati nelle fibre (MC truth)
#     2. N fotoni ai PMT (MC truth)
#     3. Efficiency vs peak threshold  (A∪B)
#     4. Efficiency vs charge threshold (A∪B)
#     5. Tabella riepilogativa
# ============================================================

import uproot as _uproot

SOURCES_WBC2 = {
    "gamma":   {
        "file":  "sim_single_wbc2_gamma_X0mm_Y0mm_Th0deg.root",
        "color": "steelblue",
        "label": r"$\gamma$ WBC2",
        "ls":    "--",
    },
    "mu_lngs": {
        "file":  "sim_single_wbc2_mu_lngs_X0mm_Y0mm_Th0deg.root",
        "color": "firebrick",
        "label": r"$\mu^-$ LNGS WBC2",
        "ls":    "--",
    },
}

# --- Digitizzazione WBC2 (stessi parametri del loop principale) ---
results_wbc2 = {}

for src_name, cfg in SOURCES_WBC2.items():
    fpath = os.path.join(DATA_DIR, cfg["file"])
    if not os.path.exists(fpath):
        print(f"[SKIP] {cfg['file']} non trovato")
        continue

    print(f"\n--- WBC2 {src_name} | {cfg['file']} ---")
    with _uproot.open(fpath) as f:
        t_ev  = f["Eventi"]
        n_gen = int(t_ev.num_entries)
        df    = f["Fotoni"].arrays(
            ["EventID","Arrival_Time_ns","PMT_ID","E_Hit_eV"], library="pd")

    if df.empty:
        print("  Nessun fotone — skip.")
        continue

    ev_ids = np.sort(df["EventID"].unique())
    print(f"  N eventi generati:       {n_gen}")
    print(f"  N eventi con fotoni PMT: {len(ev_ids)}")

    qa_list, qb_list         = [], []
    peak_a_list, peak_b_list = [], []

    for ev_id in ev_ids:
        hits = df[df["EventID"] == ev_id]
        wf = {}
        for ch in (CH_A, CH_B):
            ph_ch = hits[hits["PMT_ID"] == ch]
            if ph_ch.empty:
                t_acc = np.array([])
            else:
                wl_nm = 1240.0 / ph_ch["E_Hit_eV"].values
                t_acc = ph_ch["Arrival_Time_ns"].values[
                    np.random.rand(len(wl_nm)) < get_qe(wl_nm)]
            wf[ch] = digitize_pmt(t_acc)
        qa_list.append(charge(wf[CH_A]))
        qb_list.append(charge(wf[CH_B]))
        peak_a_list.append(float(wf[CH_A][_gate_i0:_gate_i1].min()))
        peak_b_list.append(float(wf[CH_B][_gate_i0:_gate_i1].min()))

    qa_w = np.array(qa_list); qb_w = np.array(qb_list)
    peak_a_w = np.array(peak_a_list); peak_b_w = np.array(peak_b_list)

    _df_ab = df[df["PMT_ID"].isin((CH_A, CH_B))]
    _ph_A  = _df_ab[_df_ab["PMT_ID"] == CH_A].groupby("EventID")["PMT_ID"].count()
    _ph_B  = _df_ab[_df_ab["PMT_ID"] == CH_B].groupby("EventID")["PMT_ID"].count()
    _ev_u  = _ph_A.index.union(_ph_B.index)
    n_ph_A_ev = _ph_A.reindex(_ev_u, fill_value=0).values
    n_ph_B_ev = _ph_B.reindex(_ev_u, fill_value=0).values

    results_wbc2[src_name] = {
        "qa": qa_w, "qb": qb_w,
        "peak_a": peak_a_w, "peak_b": peak_b_w,
        "n_ph_pmt": n_ph_A_ev + n_ph_B_ev,
        "n_ph_A": n_ph_A_ev, "n_ph_B": n_ph_B_ev,
        "n_gen":  n_gen,
        "n_hits": len(ev_ids),
        "cfg":    cfg,
    }

# --- Carica variabili MC truth (da Ntuple 0) per il confronto N_Cer ---
mc_truth = {}
for geo_tag, fname_pat in [("single",     "sim_single_{src}_X0mm_Y0mm_Th0deg.root"),
                            ("single_wbc2", "sim_single_wbc2_{src}_X0mm_Y0mm_Th0deg.root")]:
    for src_name in ("mu_lngs", "gamma"):
        fpath = os.path.join(DATA_DIR, fname_pat.format(src=src_name))
        if not os.path.exists(fpath):
            continue
        with _uproot.open(fpath) as f:
            t_ev = f["Eventi"]
            mc_truth[(geo_tag, src_name)] = {
                "N_cer_prod":    t_ev["N_Cerenkov_prod"].array(library="np"),
                "N_cer_entrati": t_ev["N_Cerenkov_entrati"].array(library="np"),
            }

# ============================================================
# PLOT 1: N Cherenkov prodotti e entrati nelle fibre (MC truth)
# ============================================================
fig_mc, axes_mc = plt.subplots(2, 2, figsize=(14, 10))
fig_mc.suptitle("MC truth — Cherenkov photons produced and entering the fibres\nsingle (solid) vs single_wbc2 (dashed)", fontsize=13)

for col_idx, src_name in enumerate(("mu_lngs", "gamma")):
    color_s = results[src_name]["cfg"]["color"] if src_name in results else "gray"
    label_s = results[src_name]["cfg"]["label"] if src_name in results else src_name

    for row_idx, key in enumerate(("N_cer_prod", "N_cer_entrati")):
        ax = axes_mc[row_idx, col_idx]
        title_suffix = "produced" if key == "N_cer_prod" else "entering a fibre"

        arr_s = mc_truth.get(("single", src_name), {}).get(key)
        arr_w = mc_truth.get(("single_wbc2", src_name), {}).get(key)

        for arr, ls, lbl_suf in [(arr_s, "-", "single"), (arr_w, "--", "WBC2")]:
            if arr is None:
                continue
            arr_pos = arr[arr > 0]
            if len(arr_pos) == 0:
                continue
            bins = np.logspace(np.log10(max(arr_pos.min(), 0.5)),
                               np.log10(arr_pos.max()), 60)
            ax.hist(arr_pos, bins=bins, density=True,
                    color=color_s, alpha=(0.55 if ls == "-" else 0.35),
                    histtype="stepfilled", edgecolor=color_s, lw=1.2,
                    ls=ls, label=f"{label_s}  [{lbl_suf}]  (μ={arr_pos.mean():.1f})")

        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_xlabel(f"N Cherenkov photons {title_suffix}", fontsize=10)
        ax.set_ylabel("Probability density", fontsize=10)
        ax.set_title(f"{src_name} — {title_suffix}", fontsize=11)
        ax.legend(fontsize=9); ax.grid(True, alpha=0.3, which="both")

plt.tight_layout()
fig_mc.savefig(os.path.join(OUTPUT_DIR, "wbc2_vs_single_mc_truth_cer.pdf"), bbox_inches="tight")
plt.show()

# ============================================================
# PLOT 2: N fotoni ai PMT (post-digitizzazione)
# ============================================================
fig_ph, axes_ph = plt.subplots(1, 2, figsize=(14, 5))
fig_ph.suptitle("Photons at PMTs per event — single vs single_wbc2", fontsize=13)

for ax_ph, src_name in zip(axes_ph, ("mu_lngs", "gamma")):
    color_s = results[src_name]["cfg"]["color"] if src_name in results else "gray"
    label_s = results[src_name]["cfg"]["label"] if src_name in results else src_name

    arr_s  = results.get(src_name, {}).get("n_ph_pmt")
    arr_w  = results_wbc2.get(src_name, {}).get("n_ph_pmt")

    _all = [a for a in (arr_s, arr_w) if a is not None and len(a) > 0]
    if not _all:
        continue
    x_max = max(int(a.max()) for a in _all)
    bins_ph = np.logspace(np.log10(0.8), np.log10(x_max + 1), 55)

    for arr, ls, lbl_suf, alpha in [(arr_s, "-",  "single", 0.55), (arr_w, "--", "WBC2", 0.40)]:
        if arr is None or len(arr) == 0:
            continue
        ax_ph.hist(arr, bins=bins_ph, density=True,
                   color=color_s, alpha=alpha,
                   histtype="stepfilled", edgecolor=color_s, lw=1.2, ls=ls,
                   label=f"{label_s}  [{lbl_suf}]  (n={len(arr)}, μ={arr.mean():.1f})")

    ax_ph.set_xscale("log"); ax_ph.set_yscale("log")
    ax_ph.set_xlabel("N photons at PMTs (ch A + ch B)", fontsize=11)
    ax_ph.set_ylabel("Probability density", fontsize=11)
    ax_ph.set_title(src_name, fontsize=12)
    ax_ph.legend(fontsize=9); ax_ph.grid(True, alpha=0.3, which="both")

plt.tight_layout()
fig_ph.savefig(os.path.join(OUTPUT_DIR, "wbc2_vs_single_photons_pmt.pdf"), bbox_inches="tight")
plt.show()

# ============================================================
# PLOT 3: efficienza vs soglia picco  single vs WBC2
# ============================================================
fig_ep, axes_ep = plt.subplots(1, 2, figsize=(14, 6))
fig_ep.suptitle(r"Efficiency vs peak threshold (A∪B) — single vs single_wbc2", fontsize=13)

for ax_ep, src_name in zip(axes_ep, ("mu_lngs", "gamma")):
    color_s = results[src_name]["cfg"]["color"] if src_name in results else "gray"
    label_s = results[src_name]["cfg"]["label"] if src_name in results else src_name

    for res_dict, ls, lbl_suf in [(results, "-", "single"), (results_wbc2, "--", "WBC2")]:
        res = res_dict.get(src_name)
        if res is None:
            continue
        pmin = np.minimum(res["peak_a"], res["peak_b"])
        n_h  = res["n_hits"]
        effs = np.array([100.0 * (pmin < -thr).sum() / n_h for thr in THR_AMP])
        ax_ep.plot(THR_AMP, effs, color=color_s, lw=2.0, ls=ls,
                   label=f"{label_s}  [{lbl_suf}]  (n={n_h})")

    for k in [1, 2, 3]:
        amp_k = abs(k * amp_SPE_mV)
        if amp_k <= 10.0:
            ax_ep.axvline(amp_k, color="green", lw=1.0, ls=":",
                          label=(f"1 SPE = {amp_k:.2f} mV" if k == 1 else f"{k} SPE"))
    ax_ep.axvline(abs(THR_PEAK_MV), color="gray", lw=1.2, ls="--",
                  label=f"Nominal thr = {abs(THR_PEAK_MV):.0f} mV")

    ax_ep.set_xlabel(r"|Peak threshold|  (mV)", fontsize=11)
    ax_ep.set_ylabel("Efficiency (%)", fontsize=12)
    ax_ep.set_title(src_name, fontsize=12)
    ax_ep.set_xlim(0.5, 10.0); ax_ep.set_yscale("log"); ax_ep.set_ylim(0.01, 110)
    ax_ep.legend(fontsize=9); ax_ep.grid(True, alpha=0.3, which="both")

plt.tight_layout()
fig_ep.savefig(os.path.join(OUTPUT_DIR, "wbc2_vs_single_eff_peak.pdf"), bbox_inches="tight")
plt.show()

# ============================================================
# PLOT 4: efficienza vs soglia carica  single vs WBC2
# ============================================================
fig_eq, axes_eq = plt.subplots(1, 2, figsize=(14, 6))
fig_eq.suptitle(r"Efficiency vs charge threshold (A∪B) — single vs single_wbc2", fontsize=13)

for ax_eq, src_name in zip(axes_eq, ("mu_lngs", "gamma")):
    color_s = results[src_name]["cfg"]["color"] if src_name in results else "gray"
    label_s = results[src_name]["cfg"]["label"] if src_name in results else src_name

    for res_dict, ls, lbl_suf in [(results, "-", "single"), (results_wbc2, "--", "WBC2")]:
        res = res_dict.get(src_name)
        if res is None:
            continue
        q_mx = np.maximum(res["qa"], res["qb"])
        n_h  = res["n_hits"]
        effs = np.array([100.0 * (q_mx > thr).sum() / n_h for thr in THR_Q_RANGE])
        ax_eq.plot(THR_Q_RANGE, effs, color=color_s, lw=2.0, ls=ls,
                   label=f"{label_s}  [{lbl_suf}]  (n={n_h})")

    for k in [1, 2, 3, 5, 10]:
        q_k = k * q_SPE
        if q_k <= THR_Q_MAX:
            ax_eq.axvline(q_k, color="green", lw=1.0, ls=":",
                          label=(f"1 SPE = {q_k:.0f} ADC·samples" if k == 1 else f"{k} SPE"))
    ax_eq.axvline(THR_CHARGE, color="gray", lw=1.2, ls="--",
                  label=f"Nominal ThrQ = {THR_CHARGE:.0f} ADC·samples")

    ax_eq.set_xlabel("ThrQ  (ADC·samples)", fontsize=11)
    ax_eq.set_ylabel("Efficiency (%)", fontsize=12)
    ax_eq.set_title(src_name, fontsize=12)
    ax_eq.set_xlim(0, THR_Q_MAX); ax_eq.set_yscale("log"); ax_eq.set_ylim(0.01, 110)
    ax_eq_top = ax_eq.twiny()
    ax_eq_top.set_xlim(0, 10.0)
    ax_eq_top.set_xlabel("Threshold (SPE)", fontsize=10)
    ax_eq.legend(fontsize=9); ax_eq.grid(True, alpha=0.3, which="both")

plt.tight_layout()
fig_eq.savefig(os.path.join(OUTPUT_DIR, "wbc2_vs_single_eff_charge.pdf"), bbox_inches="tight")
plt.show()

# ============================================================
# TABELLA RIEPILOGATIVA
# ============================================================
print(f"\n{'='*85}")
print(f"RIEPILOGO CONFRONTO  single  vs  single_wbc2")
print(f"Denominatore = N eventi con ≥1 fotone al PMT (n_hits)")
print(f"Soglie di riferimento: picco < {THR_PEAK_MV:.0f} mV,  carica > {THR_CHARGE:.0f} ADC·samples")
print(f"{'='*85}")
print(f"\n{'Sorgente':12} | {'Geo':11} | {'N_gen':>8} | {'N_hits':>7} | "
      f"{'acc (%)':>8} | {'⟨N_ph_PMT⟩':>11} | {'Eff_P (%)':>10} | {'Eff_Q (%)':>10}")
print("-"*95)

for src_name in ("mu_lngs", "gamma"):
    for geo_tag, res_dict in [("single", results), ("single_wbc2", results_wbc2)]:
        res = res_dict.get(src_name)
        if res is None:
            continue
        n_g  = res["n_gen"]
        n_h  = res["n_hits"]
        acc  = 100.0 * n_h / n_g
        mean_ph = res["n_ph_pmt"].mean()
        pmin = np.minimum(res["peak_a"], res["peak_b"])
        q_mx = np.maximum(res["qa"],     res["qb"])
        eff_p = 100.0 * (pmin < THR_PEAK_MV).sum()  / n_h
        eff_q = 100.0 * (q_mx > THR_CHARGE).sum()   / n_h
        print(f"{src_name:12} | {geo_tag:11} | {n_g:>8} | {n_h:>7} | "
              f"{acc:>7.3f}% | {mean_ph:>11.2f} | {eff_p:>9.2f}% | {eff_q:>9.2f}%")
    print()

print("="*85)
print("\nDelta efficienza WBC2 − single (a soglie nominali):")
for src_name in ("mu_lngs", "gamma"):
    res_s = results.get(src_name)
    res_w = results_wbc2.get(src_name)
    if res_s is None or res_w is None:
        continue
    for tag, res_t, res_r in [("single", res_s, res_s), ("wbc", res_w, res_w)]:
        pass
    def _eff_p(res):
        pmin = np.minimum(res["peak_a"], res["peak_b"])
        return 100.0 * (pmin < THR_PEAK_MV).sum() / res["n_hits"]
    def _eff_q(res):
        q_mx = np.maximum(res["qa"], res["qb"])
        return 100.0 * (q_mx > THR_CHARGE).sum() / res["n_hits"]
    dep = _eff_p(res_w) - _eff_p(res_s)
    deq = _eff_q(res_w) - _eff_q(res_s)
    print(f"  {src_name:12}:  Δeff_picco = {dep:+.2f}%   Δeff_carica = {deq:+.2f}%")
print("="*85)


In [ ]:
# ============================================================
# COINCIDENZA 2 PMT (A∩B) — single vs single_wbc2, μ e γ
#
#   Conta gli eventi con ENTRAMBI i PMT sopra soglia di picco,
#   a due soglie: ~1 SPE (−1 mV) e nominale (THR_PEAK_MV).
#   Denominatore: n_hits (eventi con ≥1 fotone al PMT).
# ============================================================

COINC_THRESHOLDS = [-1.0, THR_PEAK_MV]   # mV

_samples = []
for geo_lbl, rdict in [("single", results), ("single_wbc2", results_wbc2)]:
    for src_name in ("mu_lngs", "gamma"):
        if src_name in rdict:
            _samples.append((geo_lbl, src_name, rdict[src_name]))

for thr in COINC_THRESHOLDS:
    spe_eq = abs(thr / amp_SPE_mV)
    print(f"\n=== Soglia picco {thr:.1f} mV  (≈ {spe_eq:.1f} SPE) ===")
    print(f"{'geo':11} {'src':8} | {'n_hits':>6} | {'A∩B':>5} {'(%)':>7} | "
          f"{'solo A':>6} {'(%)':>7} | {'solo B':>6} {'(%)':>7} | {'A∪B':>5} {'(%)':>7}")
    print("-" * 100)
    for geo_lbl, src_name, res in _samples:
        A = res["peak_a"] < thr
        B = res["peak_b"] < thr
        n_hits = res["n_hits"]
        both = int((A & B).sum())
        oA   = int((A & ~B).sum())
        oB   = int((~A & B).sum())
        anyc = both + oA + oB
        print(f"{geo_lbl:11} {src_name:8} | {n_hits:>6} | "
              f"{both:>5} {100*both/n_hits:>6.1f}% | "
              f"{oA:>6} {100*oA/n_hits:>6.1f}% | "
              f"{oB:>6} {100*oB/n_hits:>6.1f}% | "
              f"{anyc:>5} {100*anyc/n_hits:>6.1f}%")

# --- Plot: frazione di eventi con 2 PMT accesi ---
fig_c, ax_c = plt.subplots(figsize=(9, 6))
fig_c.suptitle("Two-PMT coincidence fraction (A∩B)\n"
               "single vs single_wbc2 — denominator: events with ≥1 photon at PMT",
               fontsize=12)

_cats   = [(g, s) for g, s, _ in _samples]
_labels = [f"{'μ' if s == 'mu_lngs' else 'γ'} {g}" for g, s in _cats]
_x      = np.arange(len(_cats))
_wbar   = 0.35

for j, thr in enumerate(COINC_THRESHOLDS):
    fracs = []
    for _, _, res in _samples:
        both = ((res["peak_a"] < thr) & (res["peak_b"] < thr)).sum()
        fracs.append(100.0 * both / res["n_hits"])
    spe_eq = abs(thr / amp_SPE_mV)
    ax_c.bar(_x + (j - 0.5) * _wbar, fracs, _wbar,
             label=f"thr = {thr:.1f} mV (≈{spe_eq:.1f} SPE)",
             color=["#3465a4", "#cc0000"][j % 2], alpha=0.85)
    for xi, fr in zip(_x + (j - 0.5) * _wbar, fracs):
        ax_c.text(xi, fr + 1.0, f"{fr:.1f}%", ha="center", fontsize=9)

ax_c.set_xticks(_x); ax_c.set_xticklabels(_labels, fontsize=11)
ax_c.set_ylabel("Events with both PMTs above threshold (%)", fontsize=12)
ax_c.set_ylim(0, 105)
ax_c.legend(fontsize=10); ax_c.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
fig_c.savefig(os.path.join(OUTPUT_DIR, "two_pmt_coincidence_single_vs_wbc2.pdf"),
              bbox_inches="tight")
plt.show()


In [ ]:
# ============================================================
# CARICA μ con TrackL_water ≥ 10 cm  vs  spettro γ
#
#   Seleziona i muoni che hanno attraversato almeno 100 mm
#   d'acqua (TrackL_water_mm, ntuple Eventi) e sovrappone la
#   loro distribuzione di carica max(QA,QB) allo spettro γ.
#
#   Matching: la riga i-esima di "Eventi" corrisponde a
#   EventID=i (verificato: le righe con Hits>0 coincidono
#   esattamente con gli EventID presenti in "Fotoni").
# ============================================================

MIN_TRACKL_MM = 100.0   # 10 cm

# WBC3-only (results = GEO_MAIN): il pattern file e' derivato da GEO_MAIN
# cosi' e' SEMPRE coerente con `results` (niente mismatch di dimensioni).
_geoms = [(GEO_STYLE[GEO_MAIN]["tag"], results, f"sim_{GEO_MAIN}_{{s}}_X0mm_Y0mm_Th0deg.root")]
# confronto opzionale con WBC2 (decommentare dopo aver eseguito la cella di confronto):
# if "results_wbc2" in dir() and results_wbc2:
#     _geoms.append(("WBC2", results_wbc2, "sim_single_wbc2_{s}_X0mm_Y0mm_Th0deg.root"))

fig_t, axes_t = plt.subplots(1, len(_geoms), figsize=(7.5 * len(_geoms), 6), squeeze=False)
fig_t.suptitle(
    rf"Charge max(QA,QB) — $\mu^-$ with track length ≥ {MIN_TRACKL_MM/10:.0f} cm in water"
    "\n"
    r"overlaid with the $\gamma$ spectrum (all events with ≥1 photon at PMT)",
    fontsize=12)

for ax_t, (geo_lbl, rdict, fpat) in zip(axes_t[0], _geoms):
    if "mu_lngs" not in rdict or "gamma" not in rdict:
        continue

    # --- TrackL per i muoni con fotoni al PMT ---
    with uproot.open(os.path.join(DATA_DIR, fpat.format(s="mu_lngs"))) as f:
        trackl = f["Eventi"]["TrackL_water_mm"].array(library="np")
        ev_ids_mu = np.sort(np.unique(
            f["Fotoni"]["EventID"].array(library="np")).astype(int))

    trackl_hits = trackl[ev_ids_mu]                # allineato a qa/qb di results
    mask_long   = trackl_hits >= MIN_TRACKL_MM

    q_mu  = np.maximum(rdict["mu_lngs"]["qa"], rdict["mu_lngs"]["qb"])
    q_mu_sel = q_mu[mask_long]
    q_g   = np.maximum(rdict["gamma"]["qa"],   rdict["gamma"]["qb"])

    n_mu_tot = len(q_mu); n_mu_sel = int(mask_long.sum())
    print(f"[{geo_lbl}] μ con fotoni al PMT: {n_mu_tot}  →  "
          f"con TrackL ≥ {MIN_TRACKL_MM:.0f} mm: {n_mu_sel} "
          f"({100*n_mu_sel/n_mu_tot:.1f}%)   "
          f"⟨TrackL⟩ = {trackl_hits.mean():.0f} mm")
    print(f"[{geo_lbl}] ⟨max(QA,QB)⟩:  μ sel = {q_mu_sel.mean():.0f}   "
          f"γ = {q_g.mean():.1f}   (ADC·samples)")

    q_hi  = max(np.percentile(q_mu_sel, 99.0), np.percentile(q_g, 99.9))
    _bins = np.linspace(min(q_g.min(), q_mu_sel.min()), q_hi, 70)

    ax_t.hist(q_mu_sel, bins=_bins, density=True, color="firebrick", alpha=0.55,
              histtype="stepfilled", edgecolor="firebrick", lw=1.2,
              label=rf"$\mu^-$ LNGS, TrackL ≥ {MIN_TRACKL_MM/10:.0f} cm  (n={n_mu_sel})")
    ax_t.hist(q_g, bins=_bins, density=True, color="steelblue", alpha=0.55,
              histtype="stepfilled", edgecolor="steelblue", lw=1.2,
              label=rf"$\gamma$ LNGS spectrum  (n={len(q_g)})")

    for k in [1, 5, 10]:
        q_k = k * q_SPE
        if q_k <= q_hi:
            ax_t.axvline(q_k, color="green", lw=1.0, ls=":",
                         label=(f"1 SPE = {q_k:.0f} ADC·samples" if k == 1 else f"{k} SPE"))

    ax_t.set_yscale("log")
    ax_t.set_xlabel("Charge max(QA, QB)  (ADC·samples)", fontsize=11)
    ax_t.set_ylabel("Probability density", fontsize=11)
    ax_t.set_title(geo_lbl, fontsize=12)
    ax_t.legend(fontsize=9)
    ax_t.grid(True, alpha=0.3, which="both")

plt.tight_layout()
fig_t.savefig(os.path.join(OUTPUT_DIR, "charge_mu_trackL10cm_vs_gamma.pdf"),
              bbox_inches="tight")
plt.show()


In [ ]:
# === CELLA DISATTIVATA — confronto a 4 geometrie (wbc, 19f) ===
# WBC2 e' ora la geometria di riferimento, digitizzata nella cella
# 'CONFRONTO single vs single_wbc2'. Decommentare per riattivare.
# ============================================================
# # ============================================================
# # ESTENSIONE CONFRONTO: single_wbc2 (e wbc2_19f)
# #
# #   Digitizza i ROOT WBC2 con la stessa pipeline del loop
# #   principale → results_wbc2 / results_wbc2_19f, poi:
# #     - tabella riepilogativa 4 geometrie
# #     - efficienza vs soglia picco e carica (μ e γ)
# #     - coincidenza 2 PMT (A∩B)
# # ============================================================

# def _digitize_source(fname):
#     fpath = os.path.join(DATA_DIR, fname)
#     if not os.path.exists(fpath):
#         return None
#     with uproot.open(fpath) as f:
#         n_gen = int(f["Eventi"].num_entries)
#         df = f["Fotoni"].arrays(["EventID","Arrival_Time_ns","PMT_ID","E_Hit_eV"], library="pd")
#     if df.empty:
#         return None
#     ev_ids = np.sort(df["EventID"].unique())
#     qa_l, qb_l, pa_l, pb_l = [], [], [], []
#     for ev_id in ev_ids:
#         hits = df[df["EventID"] == ev_id]
#         wf = {}
#         for ch in (CH_A, CH_B):
#             ph = hits[hits["PMT_ID"] == ch]
#             if ph.empty:
#                 t_acc = np.array([])
#             else:
#                 wl = 1240.0 / ph["E_Hit_eV"].values
#                 t_acc = ph["Arrival_Time_ns"].values[np.random.rand(len(wl)) < get_qe(wl)]
#             wf[ch] = digitize_pmt(t_acc)
#         qa_l.append(charge(wf[CH_A])); qb_l.append(charge(wf[CH_B]))
#         pa_l.append(float(wf[CH_A][_gate_i0:_gate_i1].min()))
#         pb_l.append(float(wf[CH_B][_gate_i0:_gate_i1].min()))
#     _dfab = df[df["PMT_ID"].isin((CH_A, CH_B))]
#     _pA = _dfab[_dfab["PMT_ID"]==CH_A].groupby("EventID")["PMT_ID"].count()
#     _pB = _dfab[_dfab["PMT_ID"]==CH_B].groupby("EventID")["PMT_ID"].count()
#     _u  = _pA.index.union(_pB.index)
#     return {"qa": np.array(qa_l), "qb": np.array(qb_l),
#             "peak_a": np.array(pa_l), "peak_b": np.array(pb_l),
#             "n_ph_pmt": (_pA.reindex(_u, fill_value=0) + _pB.reindex(_u, fill_value=0)).values,
#             "n_gen": n_gen, "n_hits": len(ev_ids)}

# print("Digitizzazione WBC2 (può richiedere ~1 min)...")
# results_wbc2, results_wbc2_19f = {}, {}
# for src_name in ("mu_lngs", "gamma"):
#     r = _digitize_source(f"sim_single_wbc2_{src_name}_X0mm_Y0mm_Th0deg.root")
#     if r: results_wbc2[src_name] = r
#     r = _digitize_source(f"sim_single_wbc2_19f_{src_name}_X0mm_Y0mm_Th0deg.root")
#     if r: results_wbc2_19f[src_name] = r

# GEOMS = [("single", results), ("single_wbc", results_wbc),
#          ("single_wbc2", results_wbc2), ("wbc2_19f", results_wbc2_19f)]
# GEO_COLORS = {"single":"gray", "single_wbc":"darkorange",
#               "single_wbc2":"firebrick", "wbc2_19f":"purple"}
# GEO_LS     = {"single":":", "single_wbc":"--", "single_wbc2":"-", "wbc2_19f":"-."}

# # --- Tabella riepilogativa ---
# print(f"\n{'='*100}")
# print(f"CONFRONTO GEOMETRIE — soglie nominali: picco < {THR_PEAK_MV:.0f} mV, Q > {THR_CHARGE:.0f} ADC·samples")
# print(f"{'='*100}")
# print(f"{'src':8} {'geo':13} | {'N_gen':>7} | {'n_hits':>6} | {'acc%':>7} | "
#       f"{'<ph/ev>':>7} | {'eff_P%':>7} | {'eff_Q%':>7} | {'eff_P∩Q%':>8} | {'A∩B@1SPE%':>9}")
# print("-"*100)
# for src_name in ("mu_lngs", "gamma"):
#     for geo, rd in GEOMS:
#         res = rd.get(src_name)
#         if res is None: continue
#         n_h = res["n_hits"]
#         pmin = np.minimum(res["peak_a"], res["peak_b"])
#         qmx  = np.maximum(res["qa"], res["qb"])
#         effP = 100*(pmin < THR_PEAK_MV).sum()/n_h
#         effQ = 100*(qmx  > THR_CHARGE ).sum()/n_h
#         effD = 100*((pmin < THR_PEAK_MV)&(qmx > THR_CHARGE)).sum()/n_h
#         coinc = 100*((res["peak_a"] < -1.0)&(res["peak_b"] < -1.0)).sum()/n_h
#         print(f"{src_name:8} {geo:13} | {res['n_gen']:>7} | {n_h:>6} | "
#               f"{100*n_h/res['n_gen']:>6.3f}% | {res['n_ph_pmt'].mean():>7.1f} | "
#               f"{effP:>6.1f}% | {effQ:>6.1f}% | {effD:>7.1f}% | {coinc:>8.1f}%")
#     print()

# # --- Efficienza vs soglia picco e carica, 4 geometrie ---
# fig_g, axes_g = plt.subplots(2, 2, figsize=(15, 11))
# fig_g.suptitle("Efficiency vs threshold — all geometries (A∪B, denominator: n_hits)", fontsize=13)

# for col, src_name in enumerate(("mu_lngs", "gamma")):
#     ax_p, ax_q = axes_g[0][col], axes_g[1][col]
#     for geo, rd in GEOMS:
#         res = rd.get(src_name)
#         if res is None: continue
#         pmin = np.minimum(res["peak_a"], res["peak_b"])
#         qmx  = np.maximum(res["qa"], res["qb"])
#         n_h  = res["n_hits"]
#         ax_p.plot(THR_AMP, [100*(pmin < -t).sum()/n_h for t in THR_AMP],
#                   color=GEO_COLORS[geo], ls=GEO_LS[geo], lw=2,
#                   label=f"{geo} (n={n_h})")
#         ax_q.plot(THR_Q_RANGE, [100*(qmx > t).sum()/n_h for t in THR_Q_RANGE],
#                   color=GEO_COLORS[geo], ls=GEO_LS[geo], lw=2,
#                   label=f"{geo} (n={n_h})")
#     ttl = r"$\mu^-$ LNGS" if src_name == "mu_lngs" else r"$\gamma$"
#     ax_p.set_title(f"{ttl} — peak threshold", fontsize=11)
#     ax_p.set_xlabel("|Peak threshold| (mV)"); ax_p.set_xlim(0.5, 10)
#     ax_q.set_title(f"{ttl} — charge threshold", fontsize=11)
#     ax_q.set_xlabel("ThrQ (ADC·samples)");    ax_q.set_xlim(0, THR_Q_MAX)
#     for ax in (ax_p, ax_q):
#         ax.set_ylabel("Efficiency (%)"); ax.set_yscale("log"); ax.set_ylim(0.01, 110)
#         ax.legend(fontsize=9); ax.grid(True, alpha=0.3, which="both")

# plt.tight_layout()
# fig_g.savefig(os.path.join(OUTPUT_DIR, "efficiency_all_geometries.pdf"), bbox_inches="tight")
# plt.show()


In [ ]:
# ============================================================
# OTTIMIZZAZIONE 2D — single_wbc2  +  confronto Pareto con single
#
#   Replica la griglia (|thr_peak|, ThrQ) per la geometria WBC2
#   usando results_wbc2["mu_lngs"] e results_wbc2["gamma"].
#
#   Output:
#     - Heatmap eff_mu / eff_gamma / FOM  (WBC2)
#     - ROC WBC2 + sovrapposizione Pareto single vs WBC2
#     - Tabella Pareto WBC2 con rate fisiche
#     - Tabella delta Pareto (WBC2 − single a parità di ε_γ)
# ============================================================

SRC_SIG_W = "mu_lngs"
SRC_BKG_W = "gamma"

if SRC_SIG_W not in results_wbc2 or SRC_BKG_W not in results_wbc2:
    print("results_wbc2 non popolato — esegui prima la cella di confronto.")
else:
    # --- Array WBC2 ---
    pmin_sig_w = np.minimum(results_wbc2[SRC_SIG_W]["peak_a"], results_wbc2[SRC_SIG_W]["peak_b"])
    q_mx_sig_w = np.maximum(results_wbc2[SRC_SIG_W]["qa"],     results_wbc2[SRC_SIG_W]["qb"])
    n_sig_w    = results_wbc2[SRC_SIG_W]["n_hits"]
    n_gen_mu_w = results_wbc2[SRC_SIG_W]["n_gen"]

    pmin_bkg_w = np.minimum(results_wbc2[SRC_BKG_W]["peak_a"], results_wbc2[SRC_BKG_W]["peak_b"])
    q_mx_bkg_w = np.maximum(results_wbc2[SRC_BKG_W]["qa"],     results_wbc2[SRC_BKG_W]["qb"])
    n_bkg_w    = results_wbc2[SRC_BKG_W]["n_hits"]
    n_gen_gam_w = results_wbc2[SRC_BKG_W]["n_gen"]

    # Rate fisiche WBC2 (stessa configurazione della cella precedente)
    acc_mu_w  = n_sig_w  / n_gen_mu_w
    acc_gam_w = n_bkg_w  / n_gen_gam_w
    R0_mu_day_w  = RATE_MU_GEN_HZ * acc_mu_w  * SEC_PER_DAY
    R0_gam_day_w = RATE_GAMMA_HZ  * acc_gam_w * SEC_PER_DAY

    print(f"WBC2 — accettanza ottica muoni:  {acc_mu_w*100:.3f}%  "
          f"(R_mu con fotoni = {R0_mu_day_w:.3f} ev/giorno)")
    print(f"WBC2 — accettanza ottica gamma:  {acc_gam_w*100:.3f}%  "
          f"(R_gam con fotoni = {R0_gam_day_w:.1f} ev/giorno)")

    # --- Griglia (stessa risoluzione della cella single) ---
    N_PEAK_W = 60
    N_CHG_W  = 60
    thr_p_vals_w = np.linspace(0.2, 10.0,         N_PEAK_W)
    thr_q_vals_w = np.linspace(0,   10.0 * q_SPE, N_CHG_W)
    thr_p_spe_w  = thr_p_vals_w / abs(amp_SPE_mV)
    thr_q_spe_w  = thr_q_vals_w / q_SPE

    mask_p_sig_w = (pmin_sig_w[:, None] < -thr_p_vals_w[None, :]).astype(np.float32)
    mask_q_sig_w = (q_mx_sig_w[:, None] >  thr_q_vals_w[None, :]).astype(np.float32)
    mask_p_bkg_w = (pmin_bkg_w[:, None] < -thr_p_vals_w[None, :]).astype(np.float32)
    mask_q_bkg_w = (q_mx_bkg_w[:, None] >  thr_q_vals_w[None, :]).astype(np.float32)

    EFF_SIG_W = 100.0 * (mask_p_sig_w.T @ mask_q_sig_w) / n_sig_w
    EFF_BKG_W = 100.0 * (mask_p_bkg_w.T @ mask_q_bkg_w) / n_bkg_w

    RATE_MU_GRID_W  = R0_mu_day_w  * EFF_SIG_W / 100.0
    RATE_GAM_GRID_W = R0_gam_day_w * EFF_BKG_W / 100.0

    EPS_BKG = 0.1
    FOM_R_W = EFF_SIG_W / (EFF_BKG_W + EPS_BKG)
    FOM_D_W = EFF_SIG_W - EFF_BKG_W
    FOM_S_W = EFF_SIG_W / (np.sqrt(EFF_BKG_W + EPS_BKG))

    idx_R_W = np.unravel_index(FOM_R_W.argmax(), FOM_R_W.shape)
    idx_D_W = np.unravel_index(FOM_D_W.argmax(), FOM_D_W.shape)
    idx_S_W = np.unravel_index(FOM_S_W.argmax(), FOM_S_W.shape)

    print(f"\n=== PUNTI OTTIMALI — single_wbc2 ===")
    for label, idx, fom in [
        (r"ε_μ / ε_γ",  idx_R_W, FOM_R_W),
        (r"ε_μ − ε_γ",  idx_D_W, FOM_D_W),
        (r"ε_μ / √ε_γ", idx_S_W, FOM_S_W),
    ]:
        ip, iq = idx
        print(f"  FOM = {label}:")
        print(f"    |thr_peak| = {thr_p_vals_w[ip]:.2f} mV ({thr_p_spe_w[ip]:.2f} SPE)"
              f",  ThrQ = {thr_q_vals_w[iq]:.1f} ADC·samples ({thr_q_spe_w[iq]:.2f} SPE)")
        print(f"    eff_mu = {EFF_SIG_W[ip,iq]:.1f}%,  eff_gamma = {EFF_BKG_W[ip,iq]:.2f}%"
              f",  FOM = {fom[ip,iq]:.2f}")

    # =========================================================
    # PLOT 1: heatmap WBC2  (eff_mu, eff_gamma, FOM_R)
    # =========================================================
    Q_mesh_w, P_mesh_w = np.meshgrid(thr_q_spe_w, thr_p_spe_w)

    fig_hw, axes_hw = plt.subplots(1, 3, figsize=(18, 6))
    fig_hw.suptitle(
        r"2D threshold scan — single_wbc2 — signal: $\mu^-$ LNGS,  background: $\gamma$  (A∪B, P∩Q)",
        fontsize=13)

    for ax, data, title, cmap, idx_opt in [
        (axes_hw[0], EFF_SIG_W, r"Signal eff. $\epsilon_\mu$  (%)",  "Blues",   None),
        (axes_hw[1], EFF_BKG_W, r"Background eff. $\epsilon_\gamma$  (%)", "Reds",    None),
        (axes_hw[2], FOM_R_W,   r"FOM = $\epsilon_\mu\,/\,\epsilon_\gamma$", "viridis", idx_R_W),
    ]:
        im = ax.pcolormesh(Q_mesh_w, P_mesh_w, data, cmap=cmap, shading="auto")
        plt.colorbar(im, ax=ax, shrink=0.85)
        ax.set_xlabel("ThrQ  (SPE)", fontsize=11)
        ax.set_ylabel("|peak threshold|  (SPE)", fontsize=11)
        ax.set_title(title, fontsize=11)
        ax.axhline(abs(THR_PEAK_MV) / abs(amp_SPE_mV), color="white", lw=1.0, ls="--", alpha=0.7)
        ax.axvline(THR_CHARGE / q_SPE,                  color="white", lw=1.0, ls="--", alpha=0.7)
        if idx_opt is not None:
            ip, iq = idx_opt
            ax.plot(thr_q_spe_w[iq], thr_p_spe_w[ip], "w*", ms=14,
                    label=f"optimum\n({thr_p_vals_w[ip]:.1f} mV, {thr_q_spe_w[iq]:.1f} SPE)")
            ax.legend(fontsize=9)

    plt.tight_layout()
    fig_hw.savefig(os.path.join(OUTPUT_DIR, "optimization_2D_heatmap_wbc2.pdf"), bbox_inches="tight")
    plt.show()

    # =========================================================
    # PLOT 2: ROC con Pareto single (blu) vs WBC2 (rosso)
    # =========================================================

    # Pareto WBC2
    eff_s_flat_w = EFF_SIG_W.ravel()
    eff_b_flat_w = EFF_BKG_W.ravel()
    idx_sort_w   = np.argsort(eff_b_flat_w)
    g_sort_w     = eff_b_flat_w[idx_sort_w]
    m_sort_w     = eff_s_flat_w[idx_sort_w]
    pareto_mask_w = np.zeros(len(g_sort_w), dtype=bool)
    run_max_w = -np.inf
    for k in range(len(g_sort_w)):
        if m_sort_w[k] > run_max_w:
            pareto_mask_w[k] = True
            run_max_w = m_sort_w[k]

    # Ricalcola Pareto single sulla stessa griglia
    pmin_sig_s = np.minimum(results[SRC_SIG_W]["peak_a"], results[SRC_SIG_W]["peak_b"])
    q_mx_sig_s = np.maximum(results[SRC_SIG_W]["qa"],     results[SRC_SIG_W]["qb"])
    n_sig_s    = results[SRC_SIG_W]["n_hits"]
    pmin_bkg_s = np.minimum(results[SRC_BKG_W]["peak_a"], results[SRC_BKG_W]["peak_b"])
    q_mx_bkg_s = np.maximum(results[SRC_BKG_W]["qa"],     results[SRC_BKG_W]["qb"])
    n_bkg_s    = results[SRC_BKG_W]["n_hits"]

    mask_p_sig_s = (pmin_sig_s[:, None] < -thr_p_vals_w[None, :]).astype(np.float32)
    mask_q_sig_s = (q_mx_sig_s[:, None] >  thr_q_vals_w[None, :]).astype(np.float32)
    mask_p_bkg_s = (pmin_bkg_s[:, None] < -thr_p_vals_w[None, :]).astype(np.float32)
    mask_q_bkg_s = (q_mx_bkg_s[:, None] >  thr_q_vals_w[None, :]).astype(np.float32)
    EFF_SIG_S = 100.0 * (mask_p_sig_s.T @ mask_q_sig_s) / n_sig_s
    EFF_BKG_S = 100.0 * (mask_p_bkg_s.T @ mask_q_bkg_s) / n_bkg_s

    eff_s_flat_s = EFF_SIG_S.ravel()
    eff_b_flat_s = EFF_BKG_S.ravel()
    idx_sort_s   = np.argsort(eff_b_flat_s)
    g_sort_s     = eff_b_flat_s[idx_sort_s]
    m_sort_s     = eff_s_flat_s[idx_sort_s]
    pareto_mask_s = np.zeros(len(g_sort_s), dtype=bool)
    run_max_s = -np.inf
    for k in range(len(g_sort_s)):
        if m_sort_s[k] > run_max_s:
            pareto_mask_s[k] = True
            run_max_s = m_sort_s[k]

    fig_roc2, ax_roc2 = plt.subplots(figsize=(8, 7))
    fig_roc2.suptitle(
        r"Pareto comparison — single  vs  single_wbc2"
        "\n"
        r"Signal: $\mu^-$ LNGS   Background: $\gamma$ LNGS   (A∪B, P∩Q)",
        fontsize=12)

    # Scatter single (grigio sfondo)
    ax_roc2.scatter(eff_b_flat_s, eff_s_flat_s,
                    c="lightsteelblue", s=8, alpha=0.20, linewidths=0, zorder=1,
                    label="single (grid)")
    # Scatter WBC2 (colorato per FOM)
    fom_flat_w = FOM_R_W.ravel()
    vmax_w = np.percentile(fom_flat_w, 98)
    sc2 = ax_roc2.scatter(eff_b_flat_w, eff_s_flat_w,
                          c=fom_flat_w, cmap="plasma",
                          vmin=0, vmax=vmax_w,
                          s=10, alpha=0.35, linewidths=0, zorder=2,
                          label="single_wbc2 (grid)")
    plt.colorbar(sc2, ax=ax_roc2,
                 label=r"FOM_wbc2 = $\epsilon_\mu\,/\,\epsilon_\gamma$",
                 shrink=0.85)

    # Frontiera Pareto single (blu)
    p_g_s = g_sort_s[pareto_mask_s]; p_m_s = m_sort_s[pareto_mask_s]
    srt_s = np.argsort(p_g_s)
    ax_roc2.plot(p_g_s[srt_s], p_m_s[srt_s], "b-",  lw=2.5, zorder=5,
                 label="Pareto — single")
    ax_roc2.plot(p_g_s[srt_s], p_m_s[srt_s], "bo",  ms=5,   zorder=5)

    # Frontiera Pareto WBC2 (rosso)
    p_g_w = g_sort_w[pareto_mask_w]; p_m_w = m_sort_w[pareto_mask_w]
    srt_w = np.argsort(p_g_w)
    ax_roc2.plot(p_g_w[srt_w], p_m_w[srt_w], "r-",  lw=2.5, zorder=6,
                 label="Pareto — single_wbc2")
    ax_roc2.plot(p_g_w[srt_w], p_m_w[srt_w], "r^",  ms=6,   zorder=6)

    # Punto soglie correnti su entrambe le geometrie
    ip_c = np.argmin(np.abs(thr_p_vals_w - abs(THR_PEAK_MV)))
    iq_c = np.argmin(np.abs(thr_q_vals_w - THR_CHARGE))
    ax_roc2.plot(EFF_BKG_S[ip_c, iq_c], EFF_SIG_S[ip_c, iq_c],
                 "bs", ms=10, zorder=7, label="Current thresholds — single")
    ax_roc2.plot(EFF_BKG_W[ip_c, iq_c], EFF_SIG_W[ip_c, iq_c],
                 "rs", ms=10, zorder=7, label="Current thresholds — WBC2")

    ax_roc2.plot([0, 40], [0, 40], color="green", lw=0.9, ls=":", zorder=1,
                 label="Random classifier")

    ax_roc2.set_xlabel(r"Background efficiency $\epsilon_\gamma$  (%)", fontsize=12)
    ax_roc2.set_ylabel(r"Signal efficiency $\epsilon_\mu$  (%)", fontsize=12)
    ax_roc2.set_xlim(-0.5, 40); ax_roc2.set_ylim(-1, 101)
    ax_roc2.grid(True, alpha=0.3)
    ax_roc2.legend(fontsize=9, loc="lower right")
    plt.tight_layout()
    fig_roc2.savefig(os.path.join(OUTPUT_DIR, "optimization_ROC_wbc_vs_single.pdf"),
                     bbox_inches="tight")
    plt.show()

    # =========================================================
    # TABELLA PARETO — single_wbc2 con rate fisiche
    # =========================================================
    acc_mu_s  = n_sig_s  / results[SRC_SIG_W]["n_gen"]
    acc_gam_s = n_bkg_s  / results[SRC_BKG_W]["n_gen"]
    R0_mu_day_s  = RATE_MU_GEN_HZ * acc_mu_s  * SEC_PER_DAY
    R0_gam_day_s = RATE_GAMMA_HZ  * acc_gam_s * SEC_PER_DAY

    print(f"\n--- Pareto frontier — single_wbc2 ---")
    print(f"    Φ_μ = {PHI_MU_CM2_S:.0e} cm⁻²s⁻¹,  R_γ = {RATE_GAMMA_HZ:.1f} Hz")
    hdr = (f"{'ε_γ (%)':>8} | {'ε_μ (%)':>8} | {'FOM ε/ε':>8} | "
           f"{'|thr_pk|':>9} | {'ThrQ':>6} | "
           f"{'R_μ (ev/d)':>12} | {'R_γ (ev/d)':>12} | {'R_μ/R_γ':>10}")
    print(); print(hdr); print("-" * len(hdr))
    n_par_w = pareto_mask_w.sum()
    step_w  = max(1, n_par_w // 12)
    for k in np.where(pareto_mask_w)[0][::step_w]:
        g_k = g_sort_w[k]; m_k = m_sort_w[k]
        fom_k = m_k / (g_k + EPS_BKG)
        orig_k = idx_sort_w[k]
        ip_k, iq_k = np.unravel_index(orig_k, EFF_SIG_W.shape)
        r_mu_k  = R0_mu_day_w  * m_k / 100.0
        r_gam_k = R0_gam_day_w * g_k / 100.0
        ratio_k = r_mu_k / r_gam_k if r_gam_k > 0 else float("inf")
        print(f"{g_k:>8.2f} | {m_k:>8.1f} | {fom_k:>8.2f} | "
              f"{thr_p_vals_w[ip_k]:>7.2f}mV | {thr_q_spe_w[iq_k]:>4.1f}SPE | "
              f"{r_mu_k:>12.4f} | {r_gam_k:>12.2f} | {ratio_k:>10.4f}")

    # =========================================================
    # TABELLA DELTA: Δ Pareto WBC2 − single a parità di ε_γ
    # =========================================================
    print(f"\n--- Δ Pareto  WBC2 − single  (interpolato a parità di ε_γ) ---")
    print(f"{'ε_γ (%)':>8} | {'ε_μ single':>12} | {'ε_μ WBC2':>10} | "
          f"{'Δε_μ (pp)':>10} | {'ΔFOM':>8}")
    print("-"*58)

    srt_s2 = np.argsort(p_g_s); eg_s_srt = p_g_s[srt_s2]; em_s_srt = p_m_s[srt_s2]
    srt_w2 = np.argsort(p_g_w); eg_w_srt = p_g_w[srt_w2]; em_w_srt = p_m_w[srt_w2]
    eg_lo = max(eg_s_srt.min(), eg_w_srt.min())
    eg_hi = min(eg_s_srt.max(), eg_w_srt.max())
    eg_common = np.linspace(eg_lo, eg_hi, 10)
    em_s_int = np.interp(eg_common, eg_s_srt, em_s_srt)
    em_w_int = np.interp(eg_common, eg_w_srt, em_w_srt)
    for eg, em_s_i, em_w_i in zip(eg_common, em_s_int, em_w_int):
        d_em  = em_w_i - em_s_i
        d_fom = (em_w_i - em_s_i) / (eg + EPS_BKG)
        print(f"{eg:>8.2f} | {em_s_i:>12.2f} | {em_w_i:>10.2f} | "
              f"{d_em:>+10.2f} | {d_fom:>+8.3f}")
    print("="*58)
    print("(pp = punti percentuali; positivo = WBC2 migliore)")


In [ ]:
# ============================================================
# PROIEZIONE RATE — ESPERIMENTO FINALE
#
#   Box 3.5 × 3.5 × 1.5 m³; una delle due facce 3.5×3.5 ha
#   un'apertura circolare Ø 1.5 m (non instrumentata).
#
#   Ipotesi:
#   - pareti rivestite di moduli WC identici al prototipo
#     (10 cm d'acqua) → P(rivelazione | attraversamento) dal MC
#   - muoni: Φ = PHI_MU_CM2_S (Mei & Hime, superficie orizzontale),
#     distribuzione zenitale MACRO (stessa del gun_muons_lngs.mac)
#   - gamma: flusso superficiale uniforme, normalizzato a
#     RATE_GAMMA_HZ sulla superficie del prototipo → scala con l'area
#   - rate di attraversamento muoni dal metodo dell'area d'ombra
#     ⟨A_shadow(θ,φ)⟩, validato contro l'accettanza MC del prototipo
# ============================================================

# --- Box finale ---
BOX_A, BOX_B, BOX_C = 3.5, 3.5, 1.5   # m
HOLE_DIAM           = 1.5             # m (su una faccia 3.5×3.5)

# --- Soglie per la proiezione (default: nominali) ---
PROJ_THR_PEAK_MV = THR_PEAK_MV   # es. -4.0 mV
PROJ_THR_Q       = THR_CHARGE    # es. 10 ADC·samples

# --- Prototipo (acqua single module) ---
PROTO_LX, PROTO_LY, PROTO_LZ = 0.96, 0.36, 0.10   # m
PROTO_TANK = (0.98, 0.36, 0.12)                    # m (superficie esterna)

PHI_MU_M2 = PHI_MU_CM2_S * 1e4    # muoni m⁻² s⁻¹
A_GEN_M2  = 6.5 * 6.5             # piano generazione MC

# --- Distribuzione zenitale MACRO (da gun_muons_lngs.mac) ---
_th_pts = np.array([0,.0873,.1745,.2618,.3491,.4363,.5236,.6109,
                    .6981,.7854,.8727,.9599,1.0472,1.1345])
_w_pts  = np.array([3500,3300,3000,2800,2500,2200,1800,1500,
                    1300,1100,900,700,500,150.])
_thg = np.linspace(0, _th_pts[-1], 2000)
_wg  = np.interp(_thg, _th_pts, _w_pts); _wg /= _wg.sum()
_rng_proj = np.random.default_rng(42)
_N_ANG = 2_000_000
_th_s = _rng_proj.choice(_thg, size=_N_ANG, p=_wg)
_ph_s = _rng_proj.uniform(0, 2*np.pi, _N_ANG)

def _shadow(lx, ly, h):
    """⟨area d'ombra sul piano orizzontale⟩ per un box lx×ly (orizz.) × h."""
    return float(np.mean(lx*ly + np.tan(_th_s) *
                 (np.abs(np.cos(_ph_s))*ly*h + np.abs(np.sin(_ph_s))*lx*h)))

def _eff_at(res, thr_p, thr_q):
    pmin = np.minimum(res["peak_a"], res["peak_b"])
    qmx  = np.maximum(res["qa"],     res["qb"])
    n_pass = int(((pmin < thr_p) & (qmx > thr_q)).sum())
    n = res["n_hits"]
    eff = n_pass / n
    err = np.sqrt(max(n_pass, 1) * (1 - eff)) / n   # binomiale (≥1 ev per err)
    return eff, err, n_pass

# --- Validazione: area d'ombra analitica vs accettanza MC ---
A_sh_proto = _shadow(PROTO_LX, PROTO_LY, PROTO_LZ)
print("=== VALIDAZIONE metodo area d'ombra (prototipo) ===")
for lbl, rdict in [("single", results), ("single_wbc2", results_wbc2)]:
    if "mu_lngs" not in rdict: continue
    A_eff_MC = rdict["mu_lngs"]["n_hits"] / rdict["mu_lngs"]["n_gen"] * A_GEN_M2
    print(f"  {lbl:11}: A_shadow={A_sh_proto:.3f} m²  vs  A_eff MC={A_eff_MC:.3f} m²"
          f"  (ratio MC/analitica = {A_eff_MC/A_sh_proto:.3f})")

# --- Muoni: attraversamenti del box (2 orientazioni possibili) ---
A_sh_hor  = _shadow(BOX_A, BOX_B, BOX_C)   # facce 3.5×3.5 orizzontali
A_sh_vert = _shadow(BOX_A, BOX_C, BOX_B)   # facce 3.5×3.5 verticali
R_mu_hor, R_mu_vert = PHI_MU_M2*A_sh_hor, PHI_MU_M2*A_sh_vert

print(f"\n=== MUONI — rate di attraversamento del box ===")
print(f"  orizzontale (3.5×3.5 top/bottom): ⟨A_sh⟩={A_sh_hor:5.2f} m² → "
      f"{R_mu_hor*1e3:.2f} mHz = {R_mu_hor*SEC_PER_DAY:.0f} μ/giorno")
print(f"  verticale   (3.5×3.5 pareti):     ⟨A_sh⟩={A_sh_vert:5.2f} m² → "
      f"{R_mu_vert*1e3:.2f} mHz = {R_mu_vert*SEC_PER_DAY:.0f} μ/giorno")

# --- Gamma: scala di superficie ---
A_box_tot  = 2*BOX_A*BOX_B + 2*BOX_A*BOX_C + 2*BOX_B*BOX_C
A_hole     = np.pi * (HOLE_DIAM/2)**2
A_box_net  = A_box_tot - A_hole
tx, ty, tz = PROTO_TANK
A_proto_s  = 2*(tx*ty + tx*tz + ty*tz)
scale_g    = A_box_net / A_proto_s
R_g_box_hz = RATE_GAMMA_HZ * scale_g

print(f"\n=== GAMMA — scala di superficie ===")
print(f"  Box: {A_box_tot:.2f} m² − foro {A_hole:.2f} m² = {A_box_net:.2f} m²")
print(f"  Prototipo: {A_proto_s:.3f} m² → fattore = {scale_g:.1f}")
print(f"  Rate γ sul box: {R_g_box_hz:.0f} Hz   (∝ RATE_GAMMA_HZ = {RATE_GAMMA_HZ:.0f} Hz sul prototipo)")

# --- Rate rivelate ---
print(f"\n{'='*78}")
print(f"PROIEZIONE RATE RIVELATE — soglie: picco < {PROJ_THR_PEAK_MV:.1f} mV "
      f"AND Q > {PROJ_THR_Q:.0f} ADC·samples (P∩Q)")
print(f"{'='*78}")

for lbl, rdict in [("single", results), ("single_wbc2", results_wbc2)]:
    if "mu_lngs" not in rdict or "gamma" not in rdict: continue
    rm, rg = rdict["mu_lngs"], rdict["gamma"]

    eff_mu, err_mu, _   = _eff_at(rm, PROJ_THR_PEAK_MV, PROJ_THR_Q)
    eff_g,  err_g, n_pg = _eff_at(rg, PROJ_THR_PEAK_MV, PROJ_THR_Q)
    acc_g = rg["n_hits"] / rg["n_gen"]

    # P(det | attraversamento): quasi ogni μ che attraversa dà fotoni
    # (A_eff≈A_shadow) → correzione ratio, poi ε alle soglie
    ratio    = min(1.0, (rm["n_hits"]/rm["n_gen"]*A_GEN_M2) / A_sh_proto)
    p_det_mu = ratio * eff_mu

    print(f"\n  --- {lbl} ---")
    print(f"  ε_μ = {100*eff_mu:.1f} ± {100*err_mu:.1f} %"
          f"    ε_γ = {100*eff_g:.2f} ± {100*err_g:.2f} %  ({n_pg} ev γ sopra soglia)")
    for oname, rmu in [("orizzontale", R_mu_hor), ("verticale  ", R_mu_vert)]:
        print(f"  μ  ({oname}): {rmu*SEC_PER_DAY:5.0f} attr./giorno × P_det={p_det_mu:.2f}"
              f" → {rmu*SEC_PER_DAY*p_det_mu:6.0f} μ rivelati/giorno")
    r_g_det = R_g_box_hz * acc_g * eff_g
    r_g_err = R_g_box_hz * acc_g * err_g
    print(f"  γ : {R_g_box_hz:.0f} Hz × acc={100*acc_g:.3f}% × ε_γ={100*eff_g:.2f}%"
          f" → {r_g_det:.4f} ± {r_g_err:.4f} Hz = {r_g_det*SEC_PER_DAY:.0f} ± {r_g_err*SEC_PER_DAY:.0f} γ/giorno")

print(f"\nNote:")
print(f"  - il rate γ scala linearmente con RATE_GAMMA_HZ (parametro da misura reale)")
print(f"  - foro Ø{HOLE_DIAM} m: trascurato per i μ (attraversano comunque ≥1 parete),")
print(f"    sottratto dalla superficie γ sensibile")
print(f"  - statistica γ sopra soglia molto bassa: errori binomiali dominanti,")
print(f"    per ε_γ = 0 considerare il limite superiore ~ 1/n_hits")
print(f"  - muoni che attraversano 2 moduli contati una volta (rate di veto)")


# ============================================================
# TABELLA — Pareto frontier: single vs single_wbc2
#   con RATE TOTALI stimate nell'esperimento finale
#
#   Righe = soglie campionate dalla frontiera di Pareto WBC2
#   (+ riga soglie nominali). Entrambe le geometrie valutate
#   alle STESSE soglie. Richiede la cella di ottimizzazione WBC2.
#
#   R_μ/d, R_γ/d, R_tot/d = eventi/giorno rivelati nel box finale.
# ============================================================

PROJ_ORIENT = "orizzontale"   # "orizzontale" (3.5×3.5 top/bottom) | "verticale"
R_mu_cross_day = (R_mu_hor if PROJ_ORIENT == "orizzontale" else R_mu_vert) * SEC_PER_DAY

try:
    EFF_SIG_W
except NameError:
    print("\n[Tabella Pareto] Esegui prima la cella di ottimizzazione WBC2.")
else:
    # Fattori per-geometria
    _ratio_s = min(1.0, (results["mu_lngs"]["n_hits"] / results["mu_lngs"]["n_gen"]
                         * A_GEN_M2) / A_sh_proto)
    _ratio_w = min(1.0, (results_wbc2["mu_lngs"]["n_hits"] / results_wbc2["mu_lngs"]["n_gen"]
                         * A_GEN_M2) / A_sh_proto)
    _acc_g_s = results["gamma"]["n_hits"]     / results["gamma"]["n_gen"]
    _acc_g_w = results_wbc2["gamma"]["n_hits"] / results_wbc2["gamma"]["n_gen"]

    # Punti campionati dalla frontiera di Pareto WBC2 + soglie nominali
    _idx_par = np.where(pareto_mask_w)[0]
    _step    = max(1, len(_idx_par) // 14)
    _rows = []
    for k in _idx_par[::_step]:
        orig = idx_sort_w[k]
        ip, iq = np.unravel_index(orig, EFF_SIG_W.shape)
        _rows.append((ip, iq, ""))
    _ip_c = np.argmin(np.abs(thr_p_vals_w - abs(THR_PEAK_MV)))
    _iq_c = np.argmin(np.abs(thr_q_vals_w - THR_CHARGE))
    _rows.append((_ip_c, _iq_c, "  ← nominali"))

    print(f"\n--- Pareto frontier — single vs single_wbc2 — rate esperimento finale ---")
    print(f"    Box {BOX_A}×{BOX_B}×{BOX_C} m³, foro Ø{HOLE_DIAM} m — orientazione μ: {PROJ_ORIENT}")
    print(f"    μ: {R_mu_cross_day:.0f} attraversamenti/giorno;  "
          f"γ: {R_g_box_hz:.0f} Hz in superficie (∝ RATE_GAMMA_HZ = {RATE_GAMMA_HZ:.0f} Hz)")
    print(f"    Entrambe le geometrie valutate alle stesse soglie (frontiera WBC2)")
    print()
    h1 = (f"{'':>16} | {'———————————— single ————————————':^48} | "
          f"{'——————————— single_wbc2 ———————————':^48}")
    h2 = (f"{'|thr_pk|':>8} {'ThrQ':>7} | "
          f"{'ε_μ(%)':>7} {'ε_γ(%)':>7} {'R_μ/d':>8} {'R_γ/d':>10} {'R_tot/d':>10} | "
          f"{'ε_μ(%)':>7} {'ε_γ(%)':>7} {'R_μ/d':>8} {'R_γ/d':>10} {'R_tot/d':>10}")
    print(h1); print(h2); print("-" * len(h2))

    for ip, iq, tag in _rows:
        thr_p = thr_p_vals_w[ip]
        thr_q = thr_q_spe_w[iq]
        es_mu, es_g = EFF_SIG_S[ip, iq], EFF_BKG_S[ip, iq]
        ew_mu, ew_g = EFF_SIG_W[ip, iq], EFF_BKG_W[ip, iq]

        Rmu_s  = R_mu_cross_day * _ratio_s * es_mu / 100.0
        Rg_s   = R_g_box_hz * _acc_g_s * es_g / 100.0 * SEC_PER_DAY
        Rmu_w  = R_mu_cross_day * _ratio_w * ew_mu / 100.0
        Rg_w   = R_g_box_hz * _acc_g_w * ew_g / 100.0 * SEC_PER_DAY

        print(f"{thr_p:>6.2f}mV {thr_q:>4.1f}SPE | "
              f"{es_mu:>7.1f} {es_g:>7.2f} {Rmu_s:>8.1f} {Rg_s:>10.0f} {Rmu_s+Rg_s:>10.0f} | "
              f"{ew_mu:>7.1f} {ew_g:>7.2f} {Rmu_w:>8.1f} {Rg_w:>10.0f} {Rmu_w+Rg_w:>10.0f}{tag}")

    print("-" * len(h2))
    print(f"Massimi teorici (ε=100%):  R_μ = {R_mu_cross_day:.0f} μ/giorno (attraversamenti);")
    print(f"  R_γ = {R_g_box_hz*_acc_g_s*SEC_PER_DAY:.0f} (single) / "
          f"{R_g_box_hz*_acc_g_w*SEC_PER_DAY:.0f} (wbc2) γ/giorno con fotoni al PMT")
    print(f"NB: R_γ scala linearmente con RATE_GAMMA_HZ; statistica MC γ sopra soglia")
    print(f"    molto bassa alle soglie alte → valori γ indicativi (errore ~100% se 1 evento)")


In [ ]:
# ============================================================
# STUDIO READOUT — confronto fotosensori sugli stessi eventi MC
#
#   H10721-210 (PMT UBA) | S14160-6075HS (SiPM)
#   [H14603-103 fuori produzione: escluso, voce commentata nel dict]
#   Geometria: single_wbc2, sorgenti mu_lngs + gamma.
#
#   Metodo: stessi eventi Geant4, stessi numeri casuali di
#   accettazione per i 3 sensori (confronto correlato: il sensore
#   con PDE piu' alta rivela un sovrainsieme dei fotoni degli altri).
#   ⚠ Parametri da datasheet (verificare i valori esatti).
# ============================================================

SENSORS = {
    # H14603-103 (fuori produzione) — ESCLUSO dallo studio; decommentare per riferimento storico
    # "H14603-103": dict(
    #     type="pmt", gain=3.0e5, spe_res=0.40, tts_sigma=TTS_SIGMA_NS,
    #     wl =[200,250,300,350,380,400,420,450,500,550,600,650,700],
    #     eff=[0.18,0.25,0.30,0.34,0.35,0.34,0.32,0.28,0.18,0.08,0.03,0.005,0.0],
    #     color="gray", ls=":"),
    "H10721-210": dict(   # PMT ultra bialkali, HV integrato
        type="pmt", gain=2.0e6, spe_res=0.35, tts_sigma=TTS_SIGMA_NS,
        wl =[200,250,300,350,380,400,420,450,500,550,600,650,700],
        eff=[0.20,0.30,0.38,0.42,0.43,0.43,0.42,0.38,0.27,0.14,0.05,0.01,0.0],
        color="steelblue", ls="--"),
    # --- SiPM S14160-6075HS: TENUTO per confronto futuro, disattivato negli studi finali ---
    # (decommentare per rifare il confronto PMT vs SiPM)
    # "S14160-6075HS": dict(  # SiPM 6x6 mm, 75 um
    #     type="sipm", gain=5.0e6, spe_res=0.10, tts_sigma=0.15,
    #     wl =[280,300,350,400,450,500,550,600,650,700,800,900],
    #     eff=[0.10,0.18,0.32,0.44,0.50,0.46,0.40,0.33,0.27,0.21,0.10,0.04],
    #     dcr_hz=2.0e6, p_ct=0.07, tau_r=1.0, tau_f=25.0,
    #     color="firebrick", ls="-"),
}

_rng_sens = np.random.default_rng(123)
TRACE_NS  = RECORD_LENGTH * DT_NS

def _amp_pmt(cfg):
    return amp_SPE_mV * cfg["gain"] / 3.0e5     # scala con il guadagno

def _digitize_pmt_s(t_acc, gains, cfg):
    wf = np.zeros(RECORD_LENGTH)
    if len(t_acc) == 0: return wf
    jit = _rng_sens.normal(0., cfg["tts_sigma"], len(t_acc))
    ts  = t_acc + TRIGGER_OFFSET_NS + jit
    dt  = time_axis[:, None] - ts[None, :]
    return _amp_pmt(cfg) * (np.exp(-0.5*(dt/sigma_total)**2) @ gains)

def _digitize_sipm(t_acc, gains, cfg):
    # dark counts uniformi su tutta la traccia
    n_dark = _rng_sens.poisson(cfg["dcr_hz"] * TRACE_NS * 1e-9)
    t_dark = _rng_sens.uniform(0, TRACE_NS, n_dark)
    g_dark = np.maximum(0., _rng_sens.normal(1., cfg["spe_res"], n_dark))
    ts = np.concatenate([t_acc + TRIGGER_OFFSET_NS +
                         _rng_sens.normal(0., cfg["tts_sigma"], len(t_acc)), t_dark])
    gg = np.concatenate([gains, g_dark])
    if len(ts) == 0: return np.zeros(RECORD_LENGTH)
    # crosstalk: con prob p_ct un pe extra
    gg = gg * (1.0 + (_rng_sens.random(len(gg)) < cfg["p_ct"]).astype(float))
    # impulso: doppio esponenziale, carica = gain*e*R*TRANS
    Q_mVns = cfg["gain"] * e_charge * R * TRANSMISSION * 1e12   # mV*ns
    C = Q_mVns / (cfg["tau_f"] - cfg["tau_r"])
    dt = time_axis[:, None] - ts[None, :]
    shape = np.where(dt >= 0, np.exp(-np.clip(dt,0,None)/cfg["tau_f"])
                            - np.exp(-np.clip(dt,0,None)/cfg["tau_r"]), 0.0)
    return -C * (shape @ gg)

def _qspe_sensor(cfg):
    """carica di 1 SPE (ADC·samples) misurata nel gate, impulso a meta' gate."""
    t0 = np.array([0.5*(GATE_START_NS+GATE_END_NS) - TRIGGER_OFFSET_NS - 10.0])
    g  = np.ones(1)
    if cfg["type"] == "pmt":
        wf = _amp_pmt(cfg) * np.exp(-0.5*((time_axis - (t0[0]+TRIGGER_OFFSET_NS))/sigma_total)**2)
    else:
        Q_mVns = cfg["gain"] * e_charge * R * TRANSMISSION * 1e12
        C = Q_mVns / (cfg["tau_f"] - cfg["tau_r"])
        dt = time_axis - (t0[0] + TRIGGER_OFFSET_NS)
        wf = -C * np.where(dt >= 0, np.exp(-np.clip(dt,0,None)/cfg["tau_f"])
                                  - np.exp(-np.clip(dt,0,None)/cfg["tau_r"]), 0.0)
    return abs(charge(wf))

# --- caricamento eventi WBC2 (una volta sola) con numeri casuali comuni ---
sensor_results = {s: {} for s in SENSORS}
for src_name in ("mu_lngs", "gamma"):
    fpath = os.path.join(DATA_DIR, f"sim_single_wbc2_{src_name}_X0mm_Y0mm_Th0deg.root")
    if not os.path.exists(fpath):
        print(f"[SKIP] manca {fpath}"); continue
    with uproot.open(fpath) as f:
        n_gen = int(f["Eventi"].num_entries)
        df = f["Fotoni"].arrays(["EventID","Arrival_Time_ns","PMT_ID","E_Hit_eV"], library="pd")
    ev_ids = np.sort(df["EventID"].unique())
    print(f"{src_name}: {n_gen} generati, {len(ev_ids)} eventi con fotoni — digitizzo 3 sensori...")

    # prepara per evento/canale: (tempi, lambda, uniformi comuni)
    ev_data = []
    for ev_id in ev_ids:
        hits = df[df["EventID"] == ev_id]
        chd = {}
        for ch in (CH_A, CH_B):
            ph = hits[hits["PMT_ID"] == ch]
            if ph.empty:
                chd[ch] = (np.array([]), np.array([]), np.array([]))
            else:
                wl = 1240.0 / ph["E_Hit_eV"].values
                chd[ch] = (ph["Arrival_Time_ns"].values, wl,
                           _rng_sens.random(len(wl)))     # stessi u per tutti i sensori
        ev_data.append(chd)

    for sname, cfg in SENSORS.items():
        qa_l, qb_l = [], []
        for chd in ev_data:
            q_ev = {}
            for ch in (CH_A, CH_B):
                t, wl, u = chd[ch]
                acc = u < np.interp(wl, cfg["wl"], cfg["eff"], left=0., right=0.)
                gains = np.maximum(0., _rng_sens.normal(1., cfg["spe_res"], int(acc.sum())))
                wf = (_digitize_pmt_s(t[acc], gains, cfg) if cfg["type"]=="pmt"
                      else _digitize_sipm(t[acc], gains, cfg))
                q_ev[ch] = charge(wf)
            qa_l.append(q_ev[CH_A]); qb_l.append(q_ev[CH_B])
        sensor_results[sname][src_name] = {
            "qa": np.array(qa_l), "qb": np.array(qb_l),
            "n_hits": len(ev_ids), "n_gen": n_gen,
            "q_spe": _qspe_sensor(cfg)}

# --- occupanza dark del SiPM (tracce vuote) — solo se il SiPM e' attivo ---
if "S14160-6075HS" in SENSORS:
    _cfg_sipm = SENSORS["S14160-6075HS"]
    _dark_q = np.array([charge(_digitize_sipm(np.array([]), np.array([]), _cfg_sipm))
                        for _ in range(4000)])
    _qspe_sipm = _qspe_sensor(_cfg_sipm)
    p_dark_15 = (_dark_q > 1.5*_qspe_sipm).mean()
    print(f"\nSiPM dark: <PE nel gate> = {_dark_q.mean()/_qspe_sipm:.3f}, "
          f"P(Q_dark > 1.5 PE) = {100*p_dark_15:.2f}% per gate")

# --- Tabella riepilogativa ---
print(f"\n{'='*98}")
print(f"CONFRONTO SENSORI — geometria single_wbc2  (soglie in PE, carica nel gate)")
print(f"{'='*98}")
print(f"{'sensore':16} {'src':8} | {'<PE>/ev':>8} | {'eff@1PE%':>8} | {'eff@2PE%':>8} | "
      f"{'eff@3PE%':>8} | {'A∩B@1.5PE%':>10}")
print("-"*98)
for sname, cfg in SENSORS.items():
    for src_name in ("mu_lngs", "gamma"):
        r = sensor_results[sname].get(src_name)
        if r is None: continue
        qs = r["q_spe"]
        qmx = np.maximum(r["qa"], r["qb"]) / qs
        pe_mean = qmx.mean()
        coinc = 100*((r["qa"] > 1.5*qs) & (r["qb"] > 1.5*qs)).sum()/r["n_hits"]
        effs = [100*(qmx > k).sum()/r["n_hits"] for k in (1, 2, 3)]
        print(f"{sname:16} {src_name:8} | {pe_mean:>8.2f} | {effs[0]:>7.1f}% | "
              f"{effs[1]:>7.1f}% | {effs[2]:>7.1f}% | {coinc:>9.1f}%")
    print()

# --- Plot 1: spettri in PE ---
fig_s, axes_s = plt.subplots(1, 2, figsize=(14, 5.5))
fig_s.suptitle("Detected charge spectra in PE units — single_wbc2 — candidate photosensors", fontsize=12)
for ax_s, src_name in zip(axes_s, ("mu_lngs", "gamma")):
    for sname, cfg in SENSORS.items():
        r = sensor_results[sname].get(src_name)
        if r is None: continue
        qmx = np.maximum(r["qa"], r["qb"]) / r["q_spe"]
        hi = np.percentile(qmx, 99.5)
        ax_s.hist(qmx, bins=np.linspace(0, max(hi, 5), 70), density=True,
                  histtype="step", lw=2, color=cfg["color"], ls=cfg["ls"],
                  label=f"{sname}  (⟨PE⟩={qmx.mean():.1f})")
    ttl = r"$\mu^-$ LNGS" if src_name == "mu_lngs" else r"$\gamma$"
    ax_s.set_title(ttl, fontsize=11); ax_s.set_yscale("log")
    ax_s.set_xlabel("max(QA,QB)  (PE)"); ax_s.set_ylabel("Probability density")
    ax_s.legend(fontsize=9); ax_s.grid(True, alpha=0.3, which="both")
plt.tight_layout()
fig_s.savefig(os.path.join(OUTPUT_DIR, "readout_pe_spectra.pdf"), bbox_inches="tight")
plt.show()

# --- Plot 2: efficienza vs soglia in PE ---
THR_PE = np.linspace(0.2, 10, 300)
fig_e, axes_e = plt.subplots(1, 2, figsize=(14, 5.5))
fig_e.suptitle("Efficiency vs charge threshold (PE) — A∪B, denominator: n_hits", fontsize=12)
for ax_e, src_name in zip(axes_e, ("mu_lngs", "gamma")):
    for sname, cfg in SENSORS.items():
        r = sensor_results[sname].get(src_name)
        if r is None: continue
        qmx = np.maximum(r["qa"], r["qb"]) / r["q_spe"]
        ax_e.plot(THR_PE, [100*(qmx > k).sum()/r["n_hits"] for k in THR_PE],
                  color=cfg["color"], ls=cfg["ls"], lw=2, label=sname)
    ttl = r"$\mu^-$ LNGS" if src_name == "mu_lngs" else r"$\gamma$"
    ax_e.set_title(ttl, fontsize=11)
    ax_e.set_xlabel("Charge threshold (PE)"); ax_e.set_ylabel("Efficiency (%)")
    if src_name == "mu_lngs":
        ax_e.set_ylim(70, 101)          # zoom lineare sull'efficienza muoni
        ax_e.axhline(95, color="lightgray", lw=0.8, ls="--")
    else:
        ax_e.set_yscale("log"); ax_e.set_ylim(0.01, 110)
    ax_e.legend(fontsize=9); ax_e.grid(True, alpha=0.3, which="both")
plt.tight_layout()
fig_e.savefig(os.path.join(OUTPUT_DIR, "readout_efficiency_vs_pe.pdf"), bbox_inches="tight")
plt.show()


In [ ]:
# ============================================================
# TABELLA TAGLI / RATE — tre readout (soglie in PE, A∪B)
#
#   Righe = soglia di carica in PE; per ogni sensore: ε_μ, ε_γ,
#   R_μ e R_γ attese sul prototipo (ev/giorno).
#   Rate: Φ_μ Mei & Hime su piano 6.5×6.5 m; γ = RATE_GAMMA_HZ.
#   Se 0 eventi γ sopra soglia → limite superiore 90% CL (2.3/n).
#   Richiede la cella readout (sensor_results).
# ============================================================

try:
    RATE_MU_GEN_HZ, RATE_GAMMA_HZ, SEC_PER_DAY
except NameError:
    PHI_MU_CM2_S   = 3.0e-8
    RATE_MU_GEN_HZ = PHI_MU_CM2_S * (650.0)**2
    RATE_GAMMA_HZ  = 0.35/4.0 * 49400.0   # ≈4.3 kHz, da arXiv:1101.5298
    SEC_PER_DAY    = 86400.0

THR_PE_TABLE = [0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 6.0]

print(f"--- Tagli e rate prototipo — geometria single_wbc2, logica A∪B in carica ---")
print(f"    Φ_μ = 3e-8 cm⁻²s⁻¹ (piano 6.5×6.5 m),  R_γ = {RATE_GAMMA_HZ:.0f} Hz (parametrico)")
print(f"    ε_γ = '<x' → limite superiore 90% CL (0 eventi sopra soglia)")
print()

_h1 = f"{'':>9}"
_h2 = f"{'Thr (PE)':>9}"
for sname in SENSORS:
    _h1 += f" | {sname:^47}"
    _h2 += f" | {'ε_μ(%)':>8} {'ε_γ(%)':>9} {'R_μ/d':>8} {'R_γ/d':>9} {'R_μ/R_γ':>8}"
print(_h1); print(_h2); print("-" * len(_h2))

for thr in THR_PE_TABLE:
    row = f"{thr:>9.1f}"
    for sname in SENSORS:
        rm = sensor_results[sname].get("mu_lngs")
        rg = sensor_results[sname].get("gamma")
        if rm is None or rg is None:
            row += " | " + " "*47
            continue
        q_mu = np.maximum(rm["qa"], rm["qb"]) / rm["q_spe"]
        q_g  = np.maximum(rg["qa"], rg["qb"]) / rg["q_spe"]
        eff_mu = q_mu[q_mu > thr].size / rm["n_hits"]
        n_g    = int((q_g > thr).sum())
        acc_mu = rm["n_hits"] / rm["n_gen"]
        acc_g  = rg["n_hits"] / rg["n_gen"]
        R_mu = RATE_MU_GEN_HZ * acc_mu * SEC_PER_DAY * eff_mu
        if n_g > 0:
            eff_g = n_g / rg["n_hits"]
            R_g   = RATE_GAMMA_HZ * acc_g * SEC_PER_DAY * eff_g
            eg_s, rg_s = f"{100*eff_g:>8.3f}", f"{R_g:>9.1f}"
            ratio = f"{R_mu/R_g:>8.4f}" if R_g > 0 else f"{'inf':>8}"
        else:
            eff_g_ul = 2.3 / rg["n_hits"]
            R_g_ul   = RATE_GAMMA_HZ * acc_g * SEC_PER_DAY * eff_g_ul
            eg_s, rg_s = f"<{100*eff_g_ul:>7.3f}", f"<{R_g_ul:>8.1f}"
            ratio = f">{R_mu/R_g_ul:>7.4f}"
        row += f" | {100*eff_mu:>8.1f} {eg_s} {R_mu:>8.4f} {rg_s} {ratio}"
    print(row)

print("-" * len(_h2))
for sname in SENSORS:
    rm = sensor_results[sname].get("mu_lngs")
    rg = sensor_results[sname].get("gamma")
    if rm and rg:
        print(f"  {sname:16}: n_hits μ={rm['n_hits']} (N_gen={rm['n_gen']}), "
              f"γ={rg['n_hits']} (N_gen={rg['n_gen']}), q_SPE={rm['q_spe']:.0f} ADC·samples")


# ============================================================
# TABELLA 2 — rate scalate all'ESPERIMENTO FINALE
#   Box 3.5×3.5×1.5 m³ con foro Ø1.5 m (fattori dalla cella
#   PROIEZIONE: eseguirla prima). μ: attraversamenti del box ×
#   P(det|attraversamento); γ: scala di superficie.
# ============================================================
try:
    R_mu_cross_day = (R_mu_hor if PROJ_ORIENT == "orizzontale" else R_mu_vert) * SEC_PER_DAY
    _ = R_g_box_hz, A_sh_proto, A_GEN_M2
    _have_proj = True
except NameError:
    _have_proj = False
    print("\n[Tabella esperimento] Esegui prima la cella PROIEZIONE RATE (fattori geometrici del box).")

if _have_proj:
    print(f"\n--- Rate ESPERIMENTO FINALE — box 3.5×3.5×1.5 m³, orientazione μ: {PROJ_ORIENT} ---")
    print(f"    μ: {R_mu_cross_day:.0f} attraversamenti/giorno;  "
          f"γ: {R_g_box_hz:.0f} Hz in superficie (∝ RATE_GAMMA_HZ)")
    print(f"    R_γ = '<x' → limite superiore 90% CL")
    print()
    _e1 = f"{'':>9}"
    _e2 = f"{'Thr (PE)':>9}"
    for sname in SENSORS:
        _e1 += f" | {sname:^37}"
        _e2 += f" | {'R_μ/d':>9} {'R_γ/d':>12} {'R_tot/d':>12}"
    print(_e1); print(_e2); print("-" * len(_e2))

    for thr in THR_PE_TABLE:
        row = f"{thr:>9.1f}"
        for sname in SENSORS:
            rm = sensor_results[sname].get("mu_lngs")
            rg = sensor_results[sname].get("gamma")
            if rm is None or rg is None:
                row += " | " + " "*37
                continue
            q_mu = np.maximum(rm["qa"], rm["qb"]) / rm["q_spe"]
            q_g  = np.maximum(rg["qa"], rg["qb"]) / rg["q_spe"]
            eff_mu = (q_mu > thr).sum() / rm["n_hits"]
            n_g    = int((q_g > thr).sum())
            # P(det | attraversamento): quasi ogni mu che attraversa da' fotoni
            ratio  = min(1.0, (rm["n_hits"]/rm["n_gen"]*A_GEN_M2)/A_sh_proto)
            Rmu_e  = R_mu_cross_day * ratio * eff_mu
            acc_g  = rg["n_hits"] / rg["n_gen"]
            if n_g > 0:
                Rg_e = R_g_box_hz * acc_g * (n_g/rg["n_hits"]) * SEC_PER_DAY
                rg_s, rt_s = f"{Rg_e:>12.1f}", f"{Rmu_e+Rg_e:>12.1f}"
            else:
                Rg_ul = R_g_box_hz * acc_g * (2.3/rg["n_hits"]) * SEC_PER_DAY
                rg_s, rt_s = f"<{Rg_ul:>11.1f}", f"<{Rmu_e+Rg_ul:>11.1f}"
            row += f" | {Rmu_e:>9.1f} {rg_s} {rt_s}"
        print(row)
    print("-" * len(_e2))
    print("(R_tot = R_μ + R_γ rivelati; il termine γ scala linearmente con RATE_GAMMA_HZ)")


In [ ]:
# ============================================================
# SELEZIONE A DUE STADI — coincidenza bassa + recupero singolo
#
#   PASS = (QA > 1 PE  AND  QB > 1 PE)   [coincidenza]
#          OR (QA > 4 PE  OR  QB > 4 PE) [recupero canale singolo]
#
#   Applicata a: geometrie {single, wbc2} × readout {H10721-210,
#   S14160-6075HS} × sorgenti {mu_lngs, gamma}.
#   Richiede la cella readout (SENSORS, digitizzatori, _qspe_sensor).
# ============================================================

THR_COINC_PE = 2.0
THR_SINGLE_PE = 4.0
SEL_SENSORS = ["H10721-210"]   # SiPM disattivato ("S14160-6075HS" tenuto per confronto futuro)
GEO_FILES = {
    "single": "sim_single_{s}_X0mm_Y0mm_Th0deg.root",
    "wbc2":   "sim_single_wbc2_{s}_X0mm_Y0mm_Th0deg.root",
}

_rng_sens = np.random.default_rng(11)   # riproducibilità della selezione

try:
    RATE_MU_GEN_HZ, RATE_GAMMA_HZ, SEC_PER_DAY
except NameError:
    RATE_MU_GEN_HZ = 3.0e-8 * 650.0**2
    RATE_GAMMA_HZ  = 0.35/4.0 * 49400.0   # ≈4.3 kHz, da arXiv:1101.5298
    SEC_PER_DAY    = 86400.0

print(f"Selezione: (A>{THR_COINC_PE:.0f}PE ∧ B>{THR_COINC_PE:.0f}PE) ∨ "
      f"(A>{THR_SINGLE_PE:.0f}PE ∨ B>{THR_SINGLE_PE:.0f}PE)")
print(f"Rumore elettronico: {NOISE_RMS_ADC:.1f} ADC/campione → "
      f"σ_Q = {NOISE_RMS_ADC*np.sqrt(_gate_i1-_gate_i0):.1f} ADC·samples nel gate\n")
# fattori esperimento finale (dalla cella PROIEZIONE, se eseguita)
try:
    _Rmu_box_day = (R_mu_hor if PROJ_ORIENT == "orizzontale" else R_mu_vert) * SEC_PER_DAY
    _ = R_g_box_hz, A_sh_proto, A_GEN_M2
    _have_box = True
except NameError:
    _have_box = False
    print("[nota] cella PROIEZIONE non eseguita → colonna esperimento non disponibile\n")

hdr = (f"{'geo':7} {'src':8} {'sensore':14} | {'n_hits':>6} | {'coinc':>6} | "
       f"{'recup':>6} | {'PASS':>6} | {'eff%':>6} | {'rate/d (proto)':>14}"
       + (f" | {'rate/d (exp)':>13}" if _have_box else ""))
print(hdr); print("-"*len(hdr))

sel_results = {}
for geo, pat in GEO_FILES.items():
    for src_name in ("mu_lngs", "gamma"):
        fpath = os.path.join(DATA_DIR, pat.format(s=src_name))
        if not os.path.exists(fpath):
            print(f"[SKIP] {fpath}"); continue
        with uproot.open(fpath) as f:
            n_gen = int(f["Eventi"].num_entries)
            df = f["Fotoni"].arrays(["EventID","Arrival_Time_ns","PMT_ID","E_Hit_eV"],
                                    library="pd")
        ev_ids = np.sort(df["EventID"].unique())
        for sname in SEL_SENSORS:
            cfg = SENSORS[sname]
            q_spe = _qspe_sensor(cfg)
            qa_l, qb_l = [], []
            for ev_id in ev_ids:
                hits = df[df["EventID"] == ev_id]
                for ch, lst in ((CH_A, qa_l), (CH_B, qb_l)):
                    ph = hits[hits["PMT_ID"] == ch]
                    if ph.empty:
                        t_acc = np.array([])
                    else:
                        wl = 1240.0 / ph["E_Hit_eV"].values
                        acc = _rng_sens.random(len(wl)) < np.interp(
                            wl, cfg["wl"], cfg["eff"], left=0., right=0.)
                        t_acc = ph["Arrival_Time_ns"].values[acc]
                    gains = np.maximum(0., _rng_sens.normal(1., cfg["spe_res"], len(t_acc)))
                    wf = (_digitize_pmt_s(t_acc, gains, cfg) if cfg["type"] == "pmt"
                          else _digitize_sipm(t_acc, gains, cfg))
                    lst.append(charge(wf) / q_spe)
            qa, qb = np.array(qa_l), np.array(qb_l)          # pulite (PE)
            # rumore elettronico in carica: σ_Q = NOISE_RMS_ADC·√N_gate → in PE
            _sigQ_pe = NOISE_RMS_ADC * np.sqrt(_gate_i1 - _gate_i0) / q_spe
            qa_n = qa + _rng_sens.normal(0., _sigQ_pe, len(qa))
            qb_n = qb + _rng_sens.normal(0., _sigQ_pe, len(qb))
            coinc = (qa_n > THR_COINC_PE) & (qb_n > THR_COINC_PE)
            recov = ~coinc & ((qa_n > THR_SINGLE_PE) | (qb_n > THR_SINGLE_PE))
            n_pass = int((coinc | recov).sum())
            eff = n_pass / len(ev_ids)
            acc_geo = len(ev_ids) / n_gen
            r0_day = (RATE_MU_GEN_HZ if src_name == "mu_lngs" else RATE_GAMMA_HZ) \
                     * acc_geo * SEC_PER_DAY
            # scala all'esperimento finale (box 3.5×3.5×1.5 m³)
            rate_exp = None
            if _have_box:
                if src_name == "mu_lngs":
                    _ratio = min(1.0, acc_geo * A_GEN_M2 / A_sh_proto)
                    rate_exp = _Rmu_box_day * _ratio * eff
                else:
                    rate_exp = R_g_box_hz * acc_geo * SEC_PER_DAY * eff
            sel_results[(geo, src_name, sname)] = dict(
                n_hits=len(ev_ids), n_coinc=int(coinc.sum()),
                n_recov=int(recov.sum()), n_pass=n_pass, eff=eff,
                rate_day=r0_day*eff, rate_exp_day=rate_exp,
                qa=qa, qb=qb, q_spe=q_spe)   # cariche PULITE per lo scan rumore
            print(f"{geo:7} {src_name:8} {sname:14} | {len(ev_ids):>6} | "
                  f"{int(coinc.sum()):>6} | {int(recov.sum()):>6} | {n_pass:>6} | "
                  f"{100*eff:>6.1f} | {r0_day*eff:>14.1f}"
                  + (f" | {rate_exp:>13.1f}" if rate_exp is not None else ""))
    print()

print("Note:")
print(f"  - rate/d prototipo: μ da Φ Mei&Hime (piano 6.5×6.5 m), γ ∝ RATE_GAMMA_HZ={RATE_GAMMA_HZ:.0f} Hz")
print(f"  - statistica γ 'single' limitata (263 hit, 100k gen); wbc2 γ: 1M gen")
print(f"  - SiPM: le coincidenze γ includono i falsi da dark count + crosstalk")
if _have_box:
    print(f"  - rate/d (exp): box 3.5×3.5×1.5 m³ foro Ø1.5 m, orientazione μ: {PROJ_ORIENT};")
    print(f"    μ = attraversamenti box × P(det|attr.) × eff;  γ = scala di superficie (×{R_g_box_hz/RATE_GAMMA_HZ:.1f})")


In [ ]:
# ============================================================
# SCAN RUMORE — selezione a due stadi vs rumore elettronico
#
#   Usa le cariche PULITE di sensor_results (cella readout, wbc2,
#   3 sensori) e inietta rumore in carica analiticamente:
#   σ_Q = noise_ADC · √N_gate  (esatto per osservabili di carica).
#   Mostra ε_μ e ε_γ della selezione (A>1∧B>1)∨(A>4∨B>4)
#   in funzione del rumore per i tre readout.
# ============================================================

NOISE_SCAN_ADC = [0.0, 1.0, 2.0, 4.0, 6.0, 8.0]
_N_GATE = _gate_i1 - _gate_i0
_rng_ns = np.random.default_rng(77)

print(f"Selezione: (A>{THR_COINC_PE:.0f}PE ∧ B>{THR_COINC_PE:.0f}PE) ∨ "
      f"(A>{THR_SINGLE_PE:.0f}PE ∨ B>{THR_SINGLE_PE:.0f}PE) — geometria wbc2")
print(f"{'noise(ADC)':>10} | " + " | ".join(f"{s:^21}" for s in SENSORS))
print(f"{'':>10} | " + " | ".join(f"{'ε_μ(%)':>9} {'ε_γ(%)':>9}" for _ in SENSORS))
print("-" * (13 + 24*len(SENSORS)))

scan_curves = {s: {"mu": [], "g": []} for s in SENSORS}
for nadc in NOISE_SCAN_ADC:
    row = f"{nadc:>10.1f}"
    for sname in SENSORS:
        rm = sensor_results[sname].get("mu_lngs")
        rg = sensor_results[sname].get("gamma")
        if rm is None or rg is None:
            row += " | " + " "*21; continue
        effs = {}
        for tag, r in (("mu", rm), ("g", rg)):
            sigQ_pe = nadc * np.sqrt(_N_GATE) / r["q_spe"]
            qa = r["qa"]/r["q_spe"] + _rng_ns.normal(0., sigQ_pe, r["n_hits"])
            qb = r["qb"]/r["q_spe"] + _rng_ns.normal(0., sigQ_pe, r["n_hits"])
            coinc = (qa > THR_COINC_PE) & (qb > THR_COINC_PE)
            recov = ~coinc & ((qa > THR_SINGLE_PE) | (qb > THR_SINGLE_PE))
            effs[tag] = 100.0 * (coinc | recov).sum() / r["n_hits"]
            scan_curves[sname][tag].append(effs[tag])
        row += f" | {effs['mu']:>9.1f} {effs['g']:>9.1f}"
    print(row)

# --- plot ---
fig_n, (ax_m, ax_g) = plt.subplots(1, 2, figsize=(14, 5.5))
fig_n.suptitle("Two-stage selection vs electronic noise — single_wbc2\n"
               r"PASS = (A>1PE ∧ B>1PE) ∨ (A>4PE ∨ B>4PE)", fontsize=12)
for sname, cfg in SENSORS.items():
    ax_m.plot(NOISE_SCAN_ADC, scan_curves[sname]["mu"], "o-",
              color=cfg["color"], ls=cfg["ls"], lw=2, label=sname)
    ax_g.plot(NOISE_SCAN_ADC, scan_curves[sname]["g"], "o-",
              color=cfg["color"], ls=cfg["ls"], lw=2, label=sname)
ax_m.set_title(r"$\mu^-$ efficiency", fontsize=11); ax_m.set_ylim(0, 105)
ax_g.set_title(r"$\gamma$ efficiency (background!)", fontsize=11)
ax_g.set_yscale("log"); ax_g.set_ylim(0.5, 110)
for ax in (ax_m, ax_g):
    ax.set_xlabel("Noise RMS per sample (ADC counts)")
    ax.set_ylabel("Selection efficiency (%)")
    ax.axvline(2.0, color="gray", lw=1, ls="--")
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3, which="both")
ax_m.text(2.1, 10, "assumed DAQ noise", fontsize=8, color="gray")
plt.tight_layout()
fig_n.savefig(os.path.join(OUTPUT_DIR, "selection_vs_noise.pdf"), bbox_inches="tight")
plt.show()

print("\nNote:")
print("  - il rumore entra in carica come σ_Q = noise·√150 ≈ 12·noise ADC·samples:")
for sname in SENSORS:
    r = sensor_results[sname].get("mu_lngs")
    if r: print(f"    {sname:16}: q_SPE = {r['q_spe']:.0f} ADC·samples → "
                f"σ_Q(2 ADC) = {2*np.sqrt(_N_GATE)/r['q_spe']:.2f} PE")
print("  - ε_γ che CRESCE col rumore = falsi positivi da fluttuazioni sopra soglia")


In [ ]:
# ============================================================
# OTTIMIZZAZIONE IN OTTICA MUON VETO
#
#   FOM veto: massimizzare ε_μ con vincolo di deadtime:
#       deadtime = R_fake(γ) × τ_veto  ≤  budget
#
#   - ε_μ per ATTRAVERSAMENTO: ratio(geom) × ε_selezione
#   - ε_box: due pareti per attraversamento → 1−(1−ε_parete)²
#   - R_fake: rate γ sopra selezione, scala esperimento (Hz)
#   - rumore elettronico NOISE_RMS_ADC incluso analiticamente
#   Richiede: celle readout (sensor_results) e PROIEZIONE (fattori box).
# ============================================================

TAU_VETO_S     = [1e-3, 2e-3, 1e-2]   # scenari CUPID: CROSS usa 2 ms (arXiv:2510.27406)
DEADTIME_BUDGET = 0.01                 # 1%

try:
    _ = R_g_box_hz, A_sh_proto, A_GEN_M2
    _Rmu_box_day = (R_mu_hor if PROJ_ORIENT == "orizzontale" else R_mu_vert) * SEC_PER_DAY
except NameError:
    raise RuntimeError("Esegui prima la cella PROIEZIONE RATE")

_N_GATE = _gate_i1 - _gate_i0
_rng_v  = np.random.default_rng(55)

def _sel_eval(r_mu, r_g, kind, t1, t2=None):
    """kind: 'or' = A∪B>t1;  'and' = A∩B>t1;  'two' = (A∩B>t1)∨(A∪B>t2)"""
    out = {}
    for tag, r in (("mu", r_mu), ("g", r_g)):
        sig = NOISE_RMS_ADC * np.sqrt(_N_GATE) / r["q_spe"]
        qa = r["qa"]/r["q_spe"] + _rng_v.normal(0., sig, r["n_hits"])
        qb = r["qb"]/r["q_spe"] + _rng_v.normal(0., sig, r["n_hits"])
        if   kind == "or":  m = (qa > t1) | (qb > t1)
        elif kind == "and": m = (qa > t1) & (qb > t1)
        else:               m = ((qa > t1) & (qb > t1)) | (qa > t2) | (qb > t2)
        out[tag] = int(m.sum()), r["n_hits"]
    return out

print(f"OTTIMIZZAZIONE VETO — wbc2, rumore {NOISE_RMS_ADC:.1f} ADC — "
      f"R_γ_box = {R_g_box_hz:.0f} Hz (∝ RATE_GAMMA_HZ)")
print(f"ε_box = 1−(1−ε_parete)²  (2 pareti/attraversamento, foro trascurato)\n")

fig_v, ax_v = plt.subplots(figsize=(10, 7))
fig_v.suptitle("Muon-veto working points — wall μ efficiency vs absolute fake rate\n"
               "single_wbc2, electronic noise included", fontsize=12)

_best_rows = []
for sname, cfg in SENSORS.items():
    rm = sensor_results[sname].get("mu_lngs")
    rg = sensor_results[sname].get("gamma")
    if rm is None or rg is None: continue
    ratio  = min(1.0, (rm["n_hits"]/rm["n_gen"]*A_GEN_M2)/A_sh_proto)
    acc_g  = rg["n_hits"]/rg["n_gen"]

    for kind, style, lbl in [("or", "o-", "A∪B>t"), ("and", "s--", "A∩B>t")]:
        xs, ys = [], []
        for t in np.arange(0.5, 4.01, 0.25):
            res = _sel_eval(rm, rg, kind, t)
            n_mu, N_mu = res["mu"]; n_g, N_g = res["g"]
            eff_wall = ratio * n_mu / N_mu
            n_g_eff  = max(n_g, 2.3)          # limite se 0 eventi
            rfake_hz = R_g_box_hz * acc_g * n_g_eff / N_g
            xs.append(rfake_hz); ys.append(100*eff_wall)
            _best_rows.append((sname, f"{lbl.replace('t', f'{t:.2g}PE')}",
                               eff_wall, n_g, rfake_hz))
        ax_v.plot(xs, ys, style, color=cfg["color"], ms=4, lw=1.5,
                  label=f"{sname}  {lbl}", alpha=0.8)

ax_v.set_xscale("log")
ax_v.set_xlabel("Fake-veto rate from γ, experiment scale (Hz)", fontsize=12)
ax_v.set_ylabel("Wall μ efficiency per crossing (%)", fontsize=12)
ax_v.set_ylim(80, 100.5)
for tau in TAU_VETO_S:
    rmax = DEADTIME_BUDGET / tau
    ax_v.axvline(rmax, color="green", lw=1.2, ls=":")
    ax_v.text(rmax*1.1, 81, f"τ={tau*1e3:.0f} ms\n(1% dt)", fontsize=8, color="green")
ax_v.grid(True, alpha=0.3, which="both")
ax_v.legend(fontsize=9, loc="lower right")
plt.tight_layout()
fig_v.savefig(os.path.join(OUTPUT_DIR, "veto_working_points.pdf"), bbox_inches="tight")
plt.show()

# --- Tabella punti di lavoro H10721-210 ---
print(f"{'selezione':18} | {'ε_parete%':>9} | {'ε_box%':>7} | {'n_γ':>4} | "
      f"{'R_fake(Hz)':>10} | {'dt@1ms':>7} | {'dt@100ms':>8}")
print("-"*82)
for sname, sel, effw, n_g, rf in _best_rows:
    if sname != "H10721-210": continue
    eff_box = 1 - (1-effw)**2
    lim = "<" if n_g == 0 else " "
    print(f"{sel:18} | {100*effw:>9.2f} | {100*eff_box:>7.3f} | {n_g:>4} | "
          f"{lim}{rf:>9.3f} | {100*rf*1e-3:>6.2f}% | {100*rf*0.1:>7.1f}%")
print("-"*82)
print("dt = deadtime = R_fake × τ_veto;  R_fake ∝ RATE_GAMMA_HZ (da misurare!)")
print("n_γ = eventi MC γ sopra selezione (0 → limite 90% CL)")


In [ ]:
# ============================================================
# TABELLA DI DECISIONE — costo del veto in funzione di τ_veto
#
#   Per ogni selezione WC (readout scelto H10721-210) e per ogni
#   ipotesi di τ_veto:
#     - f = R_fake × τ  = frazione di esposizione persa (deadtime
#       da falsi veto = frazione di eventi LD taggati per caso)
#     - N/anno = R_LD_CUPID × f × 1 anno = eventi buoni scartati
#   più l'efficienza muone (parete e box a 2 pareti).
#
#   R_fake ∝ RATE_GAMMA_HZ (parametrico!). Rumore incluso.
#   Richiede: celle readout e PROIEZIONE.
# ============================================================

VETO_SENSOR   = "H10721-210"
TAU_VETO_MS   = [0.2, 1.0, 2.0, 10.0]      # ipotesi di finestra veto
R_LD_CUPID_HZ = 5.0                         # rate singole array CUPID
                                            # (1596 cristalli × ~3 mHz, dom. 2νββ)
SEC_PER_YEAR  = 3.156e7

_rm = sensor_results[VETO_SENSOR]["mu_lngs"]
_rg = sensor_results[VETO_SENSOR]["gamma"]
_ratio = min(1.0, (_rm["n_hits"]/_rm["n_gen"]*A_GEN_M2)/A_sh_proto)
_acc_g = _rg["n_hits"]/_rg["n_gen"]
_rng_t = np.random.default_rng(99)

def _sel_masks(r, kind, t1, t2):
    sig = NOISE_RMS_ADC*np.sqrt(_gate_i1-_gate_i0)/r["q_spe"]
    qa = r["qa"]/r["q_spe"] + _rng_t.normal(0., sig, r["n_hits"])
    qb = r["qb"]/r["q_spe"] + _rng_t.normal(0., sig, r["n_hits"])
    if   kind == "or":  return (qa > t1) | (qb > t1)
    elif kind == "and": return (qa > t1) & (qb > t1)
    else:               return ((qa > t1) & (qb > t1)) | (qa > t2) | (qb > t2)

SELEZIONI = [
    ("A∪B > 1 PE",          "or",  1.0, None),
    ("A∪B > 2 PE",          "or",  2.0, None),
    ("A∩B > 1 PE",          "and", 1.0, None),
    ("A∩B > 2 PE",          "and", 2.0, None),
    ("(A∩B>2) ∨ (>4PE)",    "two", 2.0, 4.0),
]

print(f"TABELLA DI DECISIONE VETO — sensore {VETO_SENSOR}, geometria wbc2, "
      f"rumore {NOISE_RMS_ADC:.1f} ADC")
print(f"  R_fake a scala esperimento, ∝ RATE_GAMMA_HZ = {RATE_GAMMA_HZ:.0f} Hz sul prototipo")
print(f"  R_LD CUPID = {R_LD_CUPID_HZ:.1f} Hz (array, dominata dal 2νββ)")
print(f"  ε_box = 1−(1−ε_parete)²  (2 pareti per attraversamento)")
print(f"  'f' = deadtime = frazione eventi LD persi;  'N/y' = eventi persi/anno")
print()

h1 = f"{'selezione':18} | {'ε_par':>6} | {'ε_box':>7} | {'R_fake':>8} |"
h2 = f"{'':18} | {'(%)':>6} | {'(%)':>7} | {'(Hz)':>8} |"
for tau in TAU_VETO_MS:
    h1 += f" {'τ='+f'{tau:g} ms':^18} |"
    h2 += f" {'f(%)':>8} {'N/y':>8} |"
print(h1); print(h2); print("-"*len(h2))

for lbl, kind, t1, t2 in SELEZIONI:
    m_mu = _sel_masks(_rm, kind, t1, t2)
    m_g  = _sel_masks(_rg, kind, t1, t2)
    eff_w = _ratio * m_mu.sum()/_rm["n_hits"]
    eff_b = 1 - (1-eff_w)**2
    n_g   = int(m_g.sum())
    lim   = " " if n_g > 0 else "<"
    n_g_e = max(n_g, 2.3)
    rfake = R_g_box_hz * _acc_g * n_g_e/_rg["n_hits"]
    row = (f"{lbl:18} | {100*eff_w:>6.2f} | {100*eff_b:>7.3f} | "
           f"{lim}{rfake:>7.3f} |")
    for tau in TAU_VETO_MS:
        f_dt = rfake * tau*1e-3
        n_y  = R_LD_CUPID_HZ * f_dt * SEC_PER_YEAR
        row += f" {lim}{100*f_dt:>7.4f} {lim}{n_y:>7.0f} |"
    print(row)

print("-"*len(h2))
print("Lettura: scegli la colonna del τ_veto adottato dall'esperimento; la riga")
print("migliore è quella con ε più alta il cui costo (f, N/y) resta accettabile.")
print("Se il flusso γ reale è k×100 Hz, moltiplica R_fake, f e N/y per k.")


In [ ]:
# ============================================================
# MAPPA 2D — eventi spuri/anno vs (rate cristalli, τ_veto)
#
#   N/anno = R_cristallo × N_cristalli × R_fake × τ × T_anno
#   Un pannello per selezione; ε_box (costante nel piano) nel
#   titolo; asse destro = deadtime f = R_fake×τ (dipende solo da τ).
#   ★ = ipotesi corrente (2.6 mHz/cristallo, τ=0.2 ms)
#   Richiede: celle readout e PROIEZIONE.
# ============================================================

VETO_SENSOR_2D = "H10721-210"
N_CRYSTALS     = 1596
R_CR_MHZ  = np.logspace(0, 2, 120)      # 1–100 mHz per cristallo
TAU_MS_2D = np.logspace(-1, 1.3, 120)   # 0.1–20 ms
SEC_PER_YEAR = 3.156e7

_rm = sensor_results[VETO_SENSOR_2D]["mu_lngs"]
_rg = sensor_results[VETO_SENSOR_2D]["gamma"]
_ratio = min(1.0, (_rm["n_hits"]/_rm["n_gen"]*A_GEN_M2)/A_sh_proto)
_acc_g = _rg["n_hits"]/_rg["n_gen"]
_rng_m = np.random.default_rng(99)

def _sel_eff_rfake(kind, t1, t2=None):
    out = {}
    for tag, r in (("mu", _rm), ("g", _rg)):
        sig = NOISE_RMS_ADC*np.sqrt(_gate_i1-_gate_i0)/r["q_spe"]
        qa = r["qa"]/r["q_spe"] + _rng_m.normal(0., sig, r["n_hits"])
        qb = r["qb"]/r["q_spe"] + _rng_m.normal(0., sig, r["n_hits"])
        if   kind == "or":  m = (qa > t1) | (qb > t1)
        elif kind == "and": m = (qa > t1) & (qb > t1)
        else:               m = ((qa > t1) & (qb > t1)) | (qa > t2) | (qb > t2)
        out[tag] = m.sum()
    eff_w  = _ratio * out["mu"]/_rm["n_hits"]
    rfake  = R_g_box_hz * _acc_g * max(out["g"], 2.3)/_rg["n_hits"]
    return 1-(1-eff_w)**2, rfake, int(out["g"])

SEL_2D = [("A∪B > 1 PE", "or", 1.0, None),
          ("A∩B > 1 PE", "and", 1.0, None),
          ("A∩B > 2 PE", "and", 2.0, None)]

fig_m, axes_m = plt.subplots(1, 3, figsize=(18, 5.8))
fig_m.suptitle(
    f"Falsely-tagged good events per year — {VETO_SENSOR_2D}, wbc2, "
    f"{N_CRYSTALS} crystals   (N/y = R_cr·N_cr·R_fake·τ·T;  scales with RATE_GAMMA_HZ)",
    fontsize=12)

TAUg, RCRg = np.meshgrid(TAU_MS_2D, R_CR_MHZ, indexing="ij")
for ax, (lbl, kind, t1, t2) in zip(axes_m, SEL_2D):
    eff_box, rfake, n_g = _sel_eff_rfake(kind, t1, t2)
    lim = "" if n_g > 0 else " (limite 90% CL)"
    N_grid = (RCRg*1e-3*N_CRYSTALS) * rfake * (TAUg*1e-3) * SEC_PER_YEAR
    pc = ax.pcolormesh(RCRg, TAUg, np.log10(N_grid), cmap="YlOrRd",
                       vmin=2, vmax=8, shading="auto")
    cs = ax.contour(RCRg, TAUg, N_grid,
                    levels=[1e3, 1e4, 1e5, 1e6, 1e7], colors="k", linewidths=0.9)
    ax.clabel(cs, fmt=lambda v: f"{v:.0e}/y", fontsize=8)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("Per-crystal rate (mHz)", fontsize=11)
    ax.set_ylabel(r"$\tau_{veto}$ (ms)", fontsize=11)
    ax.set_title(f"{lbl}   —   $\\epsilon_{{box}}^\\mu$ = {100*eff_box:.2f}%\n"
                 f"R_fake = {rfake:.3f} Hz{lim}", fontsize=10)
    # punto di lavoro atteso e riferimenti
    ax.plot(2.6, 0.2, "*", ms=16, color="blue", markeredgecolor="white",
            label="CUPID hypothesis\n(2.6 mHz, 0.2 ms)")
    ax.axvline(40, color="gray", lw=1, ls=":")
    ax.text(40, 0.12, " CUORE-like\n 40 mHz", fontsize=8, color="gray")
    # asse destro: deadtime (dipende solo da tau)
    ax2 = ax.twinx()
    ax2.set_yscale("log")
    ax2.set_ylim(rfake*TAU_MS_2D[0]*1e-3*100, rfake*TAU_MS_2D[-1]*1e-3*100)
    ax2.set_ylabel("deadtime f (%)", fontsize=10, color="darkred")
    ax2.tick_params(axis="y", labelcolor="darkred", labelsize=8)
    ax.legend(fontsize=8, loc="upper left")
plt.colorbar(pc, ax=axes_m, shrink=0.85, label=r"log$_{10}$(N spuri / anno)")
fig_m.savefig(os.path.join(OUTPUT_DIR, "veto_2D_spurious_map.pdf"), bbox_inches="tight")
plt.show()

print("Lettura: scegli (rate cristallo, τ_veto) → leggi N spuri/anno (contorni) e")
print("il deadtime sull'asse destro (dipende solo da τ). ε_box è nel titolo del")
print("pannello: NON varia nel piano — dipende solo dalla selezione WC.")
print(f"Tutte le mappe scalano linearmente con RATE_GAMMA_HZ (ora {RATE_GAMMA_HZ:.0f} Hz).")


In [ ]:
# ============================================================
# CONFRONTO GEOMETRIE WBC2 vs WBC3 — impatto sul veto
#
#   WBC3 = 2 layer, 45 fibre (bundle su PMT Ø8). Raccoglie ~+12%
#   di luce per muone rispetto a WBC2, MA con l'efficienza μ già
#   satura a soglia bassa il guadagno non si vede sui muoni e la
#   luce extra alza il fondo γ. Cella diagnostica: digitizza le
#   due geometrie col readout scelto e confronta le efficienze
#   di selezione. Richiede la cella readout (SENSORS, digitizer).
# ============================================================

GEO_CMP        = {"wbc2": "sim_single_wbc2_{s}_X0mm_Y0mm_Th0deg.root",
                  "wbc3": "sim_single_wbc3_{s}_X0mm_Y0mm_Th0deg.root"}
CMP_SENSOR     = "H10721-210"
_rng_cmp = np.random.default_rng(11)

def _digit_geo(fname, cfg):
    fpath = os.path.join(DATA_DIR, fname)
    if not os.path.exists(fpath):
        return None
    with uproot.open(fpath) as f:
        n_gen = int(f["Eventi"].num_entries)
        df = f["Fotoni"].arrays(["EventID","Arrival_Time_ns","PMT_ID","E_Hit_eV"], library="pd")
    ev_ids = np.sort(df["EventID"].unique())
    q_spe = _qspe_sensor(cfg)
    qa_l, qb_l = [], []
    for ev_id in ev_ids:
        hits = df[df["EventID"] == ev_id]
        for ch, lst in ((CH_A, qa_l), (CH_B, qb_l)):
            ph = hits[hits["PMT_ID"] == ch]
            if ph.empty:
                t_acc = np.array([])
            else:
                wl = 1240.0 / ph["E_Hit_eV"].values
                acc = _rng_cmp.random(len(wl)) < np.interp(wl, cfg["wl"], cfg["eff"], left=0., right=0.)
                t_acc = ph["Arrival_Time_ns"].values[acc]
            gains = np.maximum(0., _rng_cmp.normal(1., cfg["spe_res"], len(t_acc)))
            wf = (_digitize_pmt_s(t_acc, gains, cfg) if cfg["type"] == "pmt"
                  else _digitize_sipm(t_acc, gains, cfg))
            # rumore in carica (analitico, coerente con la pipeline selezione)
            lst.append(charge(wf)/q_spe + _rng_cmp.normal(0., NOISE_RMS_ADC*np.sqrt(_gate_i1-_gate_i0)/q_spe))
    return dict(qa=np.array(qa_l), qb=np.array(qb_l), n_gen=n_gen, n_hits=len(ev_ids), q_spe=q_spe)

cfg = SENSORS[CMP_SENSOR]
geo_cmp_res = {}
for geo, pat in GEO_CMP.items():
    for src_name in ("mu_lngs", "gamma"):
        r = _digit_geo(pat.format(s=src_name), cfg)
        if r: geo_cmp_res[(geo, src_name)] = r

# --- tabella efficienze di selezione ---
SELS = [("A∪B > 1 PE", "or", 1.0), ("A∩B > 1 PE", "and", 1.0),
        ("A∩B > 2 PE", "and", 2.0), ("A∩B > 3 PE", "and", 3.0)]
print(f"Readout {CMP_SENSOR}, rumore {NOISE_RMS_ADC:.0f} ADC — efficienze di selezione (A∪B)\n")
print(f"{'':22} | {'  μ (%)':^17} | {'  γ (%)':^17}")
print(f"{'selezione':22} | {'wbc2':>7} {'wbc3':>7} | {'wbc2':>7} {'wbc3':>7}")
print("-"*64)
def _eff(r, kind, t):
    qa, qb = r["qa"], r["qb"]
    m = (qa>t)|(qb>t) if kind=="or" else (qa>t)&(qb>t)
    return 100*m.sum()/r["n_hits"]
for lbl, kind, t in SELS:
    row = f"{lbl:22} |"
    for src_name in ("mu_lngs", "gamma"):
        for geo in ("wbc2", "wbc3"):
            r = geo_cmp_res.get((geo, src_name))
            row += f" {_eff(r, kind, t):>7.1f}" if r else f" {'--':>7}"
        row += " |"
    print(row)
print("-"*64)
for geo in ("wbc2", "wbc3"):
    rm = geo_cmp_res.get((geo,"mu_lngs")); rg = geo_cmp_res.get((geo,"gamma"))
    if rm and rg:
        pe_mu = np.maximum(rm["qa"],rm["qb"]).mean()
        print(f"  {geo}: ⟨PE⟩_μ = {pe_mu:.1f}  |  μ: {rm['n_hits']}/{rm['n_gen']}  γ: {rg['n_hits']}/{rg['n_gen']}")

# --- plot spettri PE sovrapposti ---
fig_c, axc = plt.subplots(1, 2, figsize=(14, 5))
fig_c.suptitle(f"WBC2 vs WBC3 — charge spectra ({CMP_SENSOR}, noise {NOISE_RMS_ADC:.0f} ADC)", fontsize=12)
cols = {"wbc2":"firebrick", "wbc3":"darkgreen"}
for ax, src_name in zip(axc, ("mu_lngs","gamma")):
    for geo in ("wbc2","wbc3"):
        r = geo_cmp_res.get((geo,src_name))
        if not r: continue
        pmax = np.maximum(r["qa"], r["qb"])
        hi = np.percentile(pmax, 99.5)
        ax.hist(pmax, bins=np.linspace(min(pmax.min(),0), max(hi,5), 70), density=True,
                histtype="step", lw=2, color=cols[geo], label=f"{geo} (⟨PE⟩={pmax.mean():.1f})")
    ax.set_title(r"$\mu^-$ LNGS" if src_name=="mu_lngs" else r"$\gamma$", fontsize=11)
    ax.set_xlabel("max(QA,QB)  (PE)"); ax.set_ylabel("Probability density")
    ax.set_yscale("log"); ax.legend(fontsize=10); ax.grid(True, alpha=0.3, which="both")
    for k in (1,2): ax.axvline(k, color="gray", lw=0.8, ls=":")
plt.tight_layout()
fig_c.savefig(os.path.join(OUTPUT_DIR, "wbc3_vs_wbc2_pe_spectra.pdf"), bbox_inches="tight")
plt.show()

print("\nConclusione: WBC3 raccoglie piu' luce (⟨PE⟩_μ piu' alto) ma con l'efficienza")
print("μ satura a soglia bassa il guadagno non si trasferisce sui muoni, mentre il")
print("fondo γ cresce. Per il veto WBC2 resta ottimale; WBC3 utile come margine ottico.")


In [ ]:
# ============================================================
# AMPIEZZA DEL SEGNALE (picco, mV) — WBC3: nuovo PMT vs SiPM
#   Distribuzioni del picco |min(wf)| per γ e μ, canale piu' attivo.
#   PMT H10721-210 (riusa digitize_pmt della cella 1) | SiPM S14160.
#   Richiede: cella 1 eseguita (digitize_pmt, get_qe, costanti).
# ============================================================
GEO_AMP = "single_wbc3"
_w0 = int(130/DT_NS); _w1 = int(260/DT_NS)          # finestra di ricerca picco
_rng_amp = np.random.default_rng(7)

# --- SiPM S14160-6075HS (solo per questo confronto di ampiezza) ---
_SIPM = dict(gain=5.0e6, spe_res=0.10, tts=0.15, dcr=2.0e6, p_ct=0.07, tau_r=1.0, tau_f=25.0,
             wl=[280,300,350,400,450,500,550,600,650,700,800,900],
             eff=[0.10,0.18,0.32,0.44,0.50,0.46,0.40,0.33,0.27,0.21,0.10,0.04])
def _sipm_qe(wl): return np.interp(wl, _SIPM["wl"], _SIPM["eff"], left=0., right=0.)
def _sipm_wf(t_acc):
    n_d = _rng_amp.poisson(_SIPM["dcr"]*RECORD_LENGTH*DT_NS*1e-9)
    ts = np.concatenate([t_acc + TRIGGER_OFFSET_NS + _rng_amp.normal(0.,_SIPM["tts"],len(t_acc)),
                         _rng_amp.uniform(0, RECORD_LENGTH*DT_NS, n_d)])
    noise = _rng_amp.normal(0., NOISE_RMS_ADC*LSB_MV, RECORD_LENGTH)
    if len(ts)==0: return noise
    g = np.maximum(0., _rng_amp.normal(1.,_SIPM["spe_res"],len(ts)))
    g = g*(1.0 + (_rng_amp.random(len(ts)) < _SIPM["p_ct"]).astype(float))
    C = _SIPM["gain"]*e_charge*R*TRANSMISSION*1e12/(_SIPM["tau_f"]-_SIPM["tau_r"])
    dt = time_axis[:,None]-ts[None,:]
    sh = np.where(dt>=0, np.exp(-np.clip(dt,0,None)/_SIPM["tau_f"])
                       - np.exp(-np.clip(dt,0,None)/_SIPM["tau_r"]), 0.0)
    return -C*(sh@g)+noise

# ampiezza 1 p.e. di riferimento (per marker)
_amp1_pmt  = abs(amp_SPE_mV)
_amp1_sipm = abs(_sipm_wf(np.array([155.0-TRIGGER_OFFSET_NS]))[_w0:_w1].min())  # ~1 pe isolato

SENS_AMP = [("H10721-210 (PMT)", digitize_pmt, get_qe, _amp1_pmt),
            ("S14160-6075HS (SiPM)", _sipm_wf, _sipm_qe, _amp1_sipm)]

amp = {name: {"gamma": [], "mu_lngs": []} for name,_,_,_ in SENS_AMP}
for s in ("gamma","mu_lngs"):
    with uproot.open(os.path.join(DATA_DIR, f"sim_{GEO_AMP}_{s}_X0mm_Y0mm_Th0deg.root")) as fh:
        df = fh["Fotoni"].arrays(["EventID","Arrival_Time_ns","PMT_ID","E_Hit_eV"], library="pd")
    for ev, h in df.groupby("EventID"):          # O(N), niente anti-pattern
        tA = h.loc[h["PMT_ID"]==CH_A]; tB = h.loc[h["PMT_ID"]==CH_B]
        for name, wf_fn, qe_fn, _ in SENS_AMP:
            pk = []
            for hc in (tA, tB):
                if len(hc)==0:
                    t = np.array([])
                else:
                    wl = 1240.0/hc["E_Hit_eV"].values
                    t  = hc["Arrival_Time_ns"].values[_rng_amp.random(len(wl)) < qe_fn(wl)]
                pk.append(wf_fn(t)[_w0:_w1].min())
            amp[name][s].append(-min(pk))        # canale piu' attivo, ampiezza positiva (mV)
for name in amp:
    for s in amp[name]: amp[name][s] = np.array(amp[name][s])

# --- plot: 2 pannelli (PMT | SiPM), γ e μ sovrapposti ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle(f"Distribuzione ampiezza picco del segnale — WBC3 (rumore {NOISE_RMS_ADC:.0f} ADC)", fontsize=13)
for ax, (name, _, _, a1) in zip(axes, SENS_AMP):
    allv = np.concatenate([amp[name]["gamma"], amp[name]["mu_lngs"]])
    hi = np.percentile(allv, 99.5)
    bins = np.linspace(0, max(hi, 5), 70)
    ax.hist(amp[name]["mu_lngs"], bins=bins, density=True, histtype="step", lw=2,
            color="#d62728", label=rf"$\mu^-$ LNGS (⟨A⟩={amp[name]['mu_lngs'].mean():.1f} mV)")
    ax.hist(amp[name]["gamma"], bins=bins, density=True, histtype="stepfilled", alpha=0.5,
            color="#1f77b4", edgecolor="#1f77b4", lw=1.5,
            label=rf"$\gamma$ LNGS (⟨A⟩={amp[name]['gamma'].mean():.1f} mV)")
    ax.axvline(a1, color="green", lw=1.2, ls=":", label=f"1 p.e. ≈ {a1:.1f} mV")
    ax.set_title(name, fontsize=12); ax.set_xlabel("|ampiezza picco|  (mV)")
    ax.set_ylabel("densità di probabilità"); ax.set_yscale("log")
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3, which="both")
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "signal_amplitude_wbc3_pmt_vs_sipm.pdf"), bbox_inches="tight")
plt.show()

print(f"WBC3 — ampiezza media picco (canale piu' attivo):")
for name in amp:
    print(f"  {name:24}: μ = {amp[name]['mu_lngs'].mean():6.1f} mV   γ = {amp[name]['gamma'].mean():5.2f} mV")
